# Notebook 08 - Deep Sequence Exploration, Freeze, And Readout

Post-Stage-0 extension separating aggressive deep-sequence exploration into three stages: **08X** (exploration lab / failure map / train-inner only), **08F** (candidate compression and freeze record), and **08O** (one-time official-validation readout of the frozen candidate). 08X must not select on official validation; 08F freezes architecture, loss, HPO, seeds, metrics, and wording rules before 08O; 08O reads official validation exactly once and never touches holdout/test.

All `RUN_08X_*`, `RUN_08F_*`, `RUN_08O_*`, `RUN_PACKAGE_BACKED_STAGE`, and `BACKUP_NOTEBOOK08_TO_GOOGLE_DRIVE` switches default to `False`. MVP: NO family is trained end-to-end inside this notebook. `last_step_lightgbm_control` emits a `fit_status="pending_last_step_lightgbm"` row that the operator overwrites once the LightGBM call is wired against the locked Stage 0 fold rows. Deep-sequence families (DLinear / TCN / GRU / LSTM / fusion) are stubbed via `NotImplementedError` and emit `failure_type="not_implemented"` rows so the schema / ledger / paper-safe-score path runs end-to-end without requiring a torch-trained model. Until those stubs are replaced, the 08O manifest is forced into `schema_only_stub=True` / `allowed_wording_bucket="no_candidate_freezable"` so empty artifacts cannot be misread as evidence.

## 08 Contract Helpers

Inline copy of canonical `src/intraday_research/contracts/deep_sequence_exploration.py` for Colab portability. Loaded BEFORE config so the config cell can reference the §5.5 constants and architecture-family enums by name.

In [ ]:
"""Artifact contract helpers for Notebook 08 outputs.

Lifts the inline validators previously embedded in
``tests/test_notebook08_artifact_contract.py`` into a reusable module so the
N08 colab notebook generator can inline the same logic for Colab portability
from this canonical package path.

The validators here mirror the design rules in
``docs/NOTEBOOK08_DEEP_SEQUENCE_EXPLORATION_FREEZE_READOUT_TECHNICAL_DESIGN_2026-06-06.md``:

* §5.5 Pre-registration Constants Table -- 13 frozen numeric thresholds.
* §7.1 architecture families enum.
* §7.9 low-compute submode enum + sub-mode B nested-fold protocol guard.
* §8.3 trial ledger schema and ``compute_tier`` enum.
* §9.1 candidate eligibility margin (numeric).
* §9.2 ``paper_safe_score`` weights/penalties.
* §9.3 fallback activation rule -- forbid official-validation-metric coupling.
* §10.1 ``OPERATOR_READOUT_AUTHORIZATION_SHA`` canonical-bytes recipe.
* §10.2 step 0 -- 08O ledger append BEFORE official-validation read.
* §13.2 freeze record schema.
* §13.3 08O manifest schema.

These helpers are validation-only utilities; they do not load, transform, or
score holdout/test data, do not import project helper packages from earlier
notebooks, and do not mount Drive.
"""

from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

import pandas as pd


NOTEBOOK08_SCOPE = "validation_only"


# ---------- §5.5 Pre-registration Constants Table ---------------------------
# Any change here MUST be accompanied by a new freeze document and a new
# 08x_search_space.json sha256 stamp (per design §5.5).

IMPROVEMENT_THRESHOLD_DELTA_MACRO_F1_LCB_95 = 0.005
IMPROVEMENT_THRESHOLD_POSITIVE_TICKER_COUNT_MIN = 4
FUSION_MIN_LCB_ADVANTAGE_OVER_COMPONENTS = 0.003
CANDIDATE_ELIGIBILITY_MIN_TRAIN_INNER_LCB_DELTA = 0.005
PAPER_SAFE_SCORE_WEIGHT_LCB_DELTA = 0.35
PAPER_SAFE_SCORE_WEIGHT_MEAN_DELTA = 0.20
PAPER_SAFE_SCORE_WEIGHT_SEED_STABILITY = 0.15
PAPER_SAFE_SCORE_WEIGHT_FOLD_CONSISTENCY = 0.10
PAPER_SAFE_SCORE_WEIGHT_PER_TICKER = 0.10
PAPER_SAFE_SCORE_PENALTY_COMPLEXITY = -0.05
PAPER_SAFE_SCORE_PENALTY_COMPUTE = -0.05
CLASS_COLLAPSE_PRED_RATE_MIN = 0.05
TOTAL_TRIAL_BUDGET_CAP_ACROSS_ALL_FAMILIES = 250

# Tier escalation thresholds (§11.1) -- quick to medium gate value.
TIER_ESCALATION_QUICK_TO_MEDIUM_LCB_DELTA_MIN = 0.003
TIER_ESCALATION_QUICK_TO_MEDIUM_POSITIVE_TICKER_MIN = 4
TIER_ESCALATION_MEDIUM_TO_AGGRESSIVE_SEED_STD_MAX = 0.01

# §9.2 fallback scale when N06 seed-std artifact is unavailable.
SEED_STABILITY_SCALE_FALLBACK = 0.02


# ---------- §7.1 Architecture Families --------------------------------------

ARCHITECTURE_FAMILIES = (
    "ms_dlinear_tcn",
    "dlinear_only",
    "tcn_only",
    "shallow_gru",
    "shallow_lstm",
    "last_step_mlp_sequence_ablation",
    "last_step_lightgbm_control",
)

# Families currently allowed inside an 08X search-space payload. GRU/LSTM remain
# section-7.1 candidate families, but they are withheld from 08X until their
# axis blocks are frozen in config/search-space and sha-stamped before trial 0.
SEARCH_ELIGIBLE_ARCHITECTURE_FAMILIES = (
    "ms_dlinear_tcn",
    "dlinear_only",
    "tcn_only",
    "last_step_mlp_sequence_ablation",
    "last_step_lightgbm_control",
)


# ---------- §7.6 HPO Methods + §7.9 Low-Compute -----------------------------

HPO_METHODS = (
    "random_search",
    "tpe",
    "successive_halving",
    "asha",
    "hyperband",
    "bohb",
)

LOW_COMPUTE_SUBMODES = ("deterministic_agg", "train_inner_oof_mlp_head")

# Sub-mode B nested-fold protocol -- required search-space fields (§7.9).
LOW_COMPUTE_SUBMODE_B_REQUIRED_FIELDS = (
    "outer_fold_scheme",
    "outer_fold_k",
    "inner_fold_k_for_head",
    "head_train_data_source",
    "head_eval_data_source",
)
LOW_COMPUTE_SUBMODE_B_ALLOWED_OUTER_FOLD_SCHEMES = (
    "rolling_origin_folds",
    "purged_time_series_folds",
    "embargoed_train_inner_folds",
)
LOW_COMPUTE_SUBMODE_B_MIN_OUTER_FOLD_K = 5
LOW_COMPUTE_SUBMODE_B_MIN_INNER_FOLD_K = 5


# ---------- §8.3 Trial Ledger ----------------------------------------------

COMPUTE_TIER_VALUES = ("full_compute", "low_compute")

REQUIRED_TRIAL_LEDGER_COLUMNS = {
    "trial_id",
    "candidate_family",
    "candidate_id",
    "config_hash",
    "fold_id",
    "seed",
    "budget_tier",
    "max_epochs",
    "actual_epochs",
    "early_stop_reason",
    "fit_status",
    "failure_type",
    "failure_message",
    "train_inner_fit_n",
    "train_inner_validation_n",
    "macro_f1",
    "balanced_accuracy",
    "accuracy",
    "stratified_dummy_macro_f1_same_rows",
    "delta_macro_f1_vs_dummy",
    "class0_pred_rate",
    "class1_pred_rate",
    "ticker_max_share",
    "actual_wall_clock_seconds",
    "peak_memory_mb",
    "gpu_seconds_or_null",
    "compute_tier",
    "scope",
    "official_validation_used",
    "holdout_test_authorized",
}


# §8.3 fit_status enum -- consumed by validate_trial_ledger_frame.
# "pending" / "pending_last_step_lightgbm" are pre-fit MVP placeholders that
# the operator overwrites with one of the post-fit statuses before treating a
# row as evidence; "completed" and "failed" are the post-fit terminal states;
# "skipped" covers trials a tier-escalation gate refuses without running.
FIT_STATUSES = (
    "pending",
    "pending_last_step_lightgbm",
    "completed",
    "failed",
    "skipped",
)


# §8.4 failure-map enum (used by 08X failure-row writer).
FAILURE_TYPES = (
    "class_collapse",
    "unstable_seed_variance",
    "ticker_concentration",
    "date_concentration",
    "time_of_day_concentration",
    "training_divergence",
    "timeout",
    "memory_error",
    "artifact_schema_failure",
    "official_validation_boundary_violation",
    "feature_window_leak_detected",
    "insufficient_same_row_dummy",
    "no_improvement_over_last_step_control",
    # MVP-only placeholder; deep families fit via NotImplementedError -> ledger row.
    "not_implemented",
)


# ---------- DMC Attestation (§9.1) ------------------------------------------

REQUIRED_DMC_FIELDS = {
    "dmc_role",
    "reviewer_identifier",
    "reviewed_08x_run_manifest_sha256",
    "reviewed_at_utc",
    "attestation_statement",
}


# ---------- Freeze Record (§13.2) -------------------------------------------

REQUIRED_FREEZE_RECORD_FIELDS = {
    "stage",
    "scope",
    "primary_candidate_id",
    "fallback_candidate_id",
    "fallback_activation_rule",
    "config_hash",
    "architecture_family",
    "frozen_architecture_params",
    "frozen_loss",
    "frozen_hpo_method",
    "frozen_seed_list",
    "frozen_metric_list",
    "frozen_wording_rule",
    "paper_safe_score",
    "official_validation_used_for_selection",
    "holdout_test_authorized",
}

# §13.2 optional but recommended freeze record fields. The contract does NOT
# require these at write time (so the test fixtures stay minimal) but the
# generator writes them by default.
RECOMMENDED_FREEZE_RECORD_FIELDS = {
    "paper_safe_score_runner_up",
    "paper_safe_score_margin",
    "challenger_baseline_id",
    "frozen_code_git_sha",
    "frozen_python_env_hash",
    "frozen_dependency_versions",
    "low_compute_baseline",
    "low_compute_submode",
}


# Belt-and-suspenders: exact-substring layer catches snake_case identifiers
# that survive normalization without separator collapse.
FORBIDDEN_FALLBACK_RULE_SUBSTRINGS = (
    "official_validation_macro_f1",
    "official_validation_delta",
    "official_val_score",
    "official_validation_balanced_accuracy",
    "official_val_metric",
)


# Regex layer catches English-prose abuses across separators. Patterns are
# applied to the normalized rule string (lowercase + [-_\s]+ collapsed to " ").
FORBIDDEN_FALLBACK_PATTERNS_NORMALIZED = (
    # "official val ... <metric>"
    r"\bofficial val(?:idation)?\b[^.]*?\b(?:macro f1|balanced accuracy|delta|f1 score|f1|accuracy|metric|result|performance|auc|loss)\b",
    # "<metric> ... official val"   (note: 'score' alone excluded so "scoring official val" is OK)
    r"\b(?:macro f1|balanced accuracy|delta|f1 score|metric|result|performance|auc|loss)\b[^.]*?\bofficial val(?:idation)?\b",
    # "primary scor(es/ing) ... wors/lower/fail/poorly" -- primary metric comparison
    r"\b(?:primary|fallback|model|deep)\b[^.]*?\b(?:scor|perform)[a-z]*\b[^.]*?\b(?:wors|worse|lower|low|fail|poorly|badly)\b",
    # explicit comparison verbs
    r"\b(?:fails? to beat|cannot beat|does not beat|outperformed by|beaten by|loses to)\b",
)


# ---------- §10.1 OPERATOR_READOUT_AUTHORIZATION_SHA inputs ----------------

OPERATOR_READOUT_AUTHORIZATION_INPUTS = (
    ("08f_candidate_freeze_record.json", "json_canonical"),
    (
        "docs/NOTEBOOK08_DEEP_SEQUENCE_EXPLORATION_FREEZE_READOUT_TECHNICAL_DESIGN_2026-06-06.md",
        "text_lf",
    ),
    ("AGENTS.md", "text_lf"),
)


# ---------- §13.1/§13.3 08X / 08O Manifest Fields --------------------------

REQUIRED_08X_RUN_MANIFEST_FIELDS = {
    "notebook08_version",
    "stage",
    "scope",
    "source_stage0_candidate",
    "official_validation_used",
    "holdout_test_authorized",
    "train_inner_fold_policy",
    "purge_policy",
    "embargo_policy",
    "search_budget_tier",
    "trial_count_requested",
    "trial_count_completed",
    "trial_count_failed",
    "trial_count_skipped",
}

REQUIRED_08O_RUN_MANIFEST_FIELDS = {
    "stage",
    "scope",
    "primary_candidate_id",
    "freeze_record_sha256",
    "official_validation_readout_started_at",
    "official_validation_used_for_selection",
    "holdout_test_authorized",
    "same_row_dummy_present",
    "per_ticker_present",
    "seed_summary_present",
    "allowed_wording_bucket",
}


# ---------- §13.3 08O artifact completeness gate (Round 8 #1) --------------
# Round 7 hardened the MANIFEST so a schema_only_stub manifest cannot carry
# evidence wording. Round 8 closes the upstream gap: the generator's previous
# "wrote_any_rows = any(file non-empty)" check was too permissive -- writing
# one row to one artifact would flip the manifest into real-readout mode.
#
# Real-mode is now defined as: every REQUIRED artifact present + non-empty +
# carries the §13.3 schema columns. Anything short of that stays in stub
# mode, which §10.4 forces into the no_candidate_freezable wording bucket.
#
# 08o_concentration_guardrails.csv and 08o_failure_rows.csv are intentionally
# NOT in REQUIRED_08O_REAL_READOUT_ARTIFACTS -- a real readout with zero
# concentration warnings and zero failed seeds is legitimate, so requiring
# non-empty rows there would create a false-positive stub flag. They remain
# in OUTPUT_FILES_08O for write-out but do not gate real-mode promotion.

REQUIRED_08O_REAL_READOUT_ARTIFACTS = {
    "08o_validation_readout.csv": {
        "seed",
        "macro_f1",
        "balanced_accuracy",
        "accuracy",
        "delta_macro_f1_vs_stratified_dummy_same_rows",
        "delta_balanced_accuracy_vs_stratified_dummy_same_rows",
        "validation_n",
        "class0_pred_rate",
        "class1_pred_rate",
    },
    "08o_validation_per_ticker.csv": {
        "ticker",
        "macro_f1",
        "delta_macro_f1_vs_dummy",
        "n_rows",
    },
    "08o_seed_summary.csv": {
        "metric",
        "seed_mean",
        "seed_std",
        "seed_lcb_95",
    },
    "08o_same_row_baselines.csv": {
        "baseline",
        "macro_f1_mean",
        "macro_f1_std",
    },
}


# ---------- §13.2 08F Freeze MD/JSON Output Filenames ----------------------

OUTPUT_FILES_08X = (
    "08x_search_space.json",
    "08x_trial_ledger.csv",
    "08x_fold_results.csv",
    "08x_seed_summary.csv",
    "08x_failure_ledger.csv",
    "08x_candidate_compression_table.csv",
    "08x_run_manifest.json",
    "08x_environment_manifest.json",
)

OUTPUT_FILES_08F = (
    "08f_candidate_freeze_record.json",
    "08f_candidate_freeze_record.md",
    "08f_candidate_compression_audit.csv",
    "08f_static_gate_report.json",
    "08f_no_candidate_freezable.json",
)

OUTPUT_FILES_08O = (
    "08o_validation_readout.csv",
    "08o_validation_per_ticker.csv",
    "08o_seed_summary.csv",
    "08o_same_row_baselines.csv",
    "08o_concentration_guardrails.csv",
    "08o_failure_rows.csv",
    "08o_decision_record.json",
    "08o_run_manifest.json",
)


# ---------- §10.4 Allowed Wording Buckets ----------------------------------

ALLOWED_WORDING_BUCKETS = (
    "improvement",
    "weak_mixed",
    "low_compute_baseline",
    "no_candidate_freezable",
    "unstable",
)


# ===========================================================================
# Helpers
# ===========================================================================


def _normalize_fallback_rule(text: str) -> str:
    """Lowercase + collapse [-_\\s] runs to single space so prose variants
    ('official-validation', 'official_val', 'official validation') normalize
    to the same form before regex matching."""
    return re.sub(r"[\s\-_]+", " ", text.lower())


def operator_readout_authorization_sha(
    fixed_order_inputs: list[tuple[Path, str]],
) -> str:
    """Compute the §10.1 canonical-bytes sha256 over fixed-order inputs.

    Recipe (verbatim from design §10.1):
      for each input in fixed_order_inputs (DO NOT sort, DO NOT change order):
        1. encode the input's relative_path as UTF-8 bytes;
        2. emit `len(path_bytes).to_bytes(8, "big") + path_bytes`;
        3. canonicalize the file bytes:
             - "json_canonical": json.loads(text) then
               json.dumps(obj, sort_keys=True, separators=(",", ":"),
               ensure_ascii=False, allow_nan=False).encode("utf-8");
             - "text_lf": open(path, "rb").read().replace(b"\\r\\n", b"\\n");
        4. emit `len(canonical_bytes).to_bytes(8, "big") + canonical_bytes`.
      return hashlib.sha256(stream).hexdigest()

    ``fixed_order_inputs`` is a list of ``(Path, mode)`` where mode is
    ``"json_canonical"`` or ``"text_lf"``. ``Path`` may be absolute (the
    canonical bytes use ``Path.as_posix()`` relative to the project root,
    so callers must pass already-relative paths; this matches how design
    §10.1 specifies "relative_path").
    """
    hasher = hashlib.sha256()
    for path, mode in fixed_order_inputs:
        rel_path = str(path).replace("\\", "/")
        path_bytes = rel_path.encode("utf-8")
        hasher.update(len(path_bytes).to_bytes(8, "big"))
        hasher.update(path_bytes)
        raw_bytes = Path(path).read_bytes()
        if mode == "json_canonical":
            obj = json.loads(raw_bytes.decode("utf-8"))
            canonical = json.dumps(
                obj,
                sort_keys=True,
                separators=(",", ":"),
                ensure_ascii=False,
                allow_nan=False,
            ).encode("utf-8")
        elif mode == "text_lf":
            canonical = raw_bytes.replace(b"\r\n", b"\n")
        else:
            raise ValueError(
                f"unsupported canonicalization mode: {mode!r} "
                f"(expected 'json_canonical' or 'text_lf')"
            )
        hasher.update(len(canonical).to_bytes(8, "big"))
        hasher.update(canonical)
    return hasher.hexdigest()


# ===========================================================================
# Validators
# ===========================================================================


def validate_dmc_attestation(payload: dict) -> None:
    """§9.1 dmc_attestation.json schema -- required fields, role enum, hex sha256."""
    missing = REQUIRED_DMC_FIELDS - payload.keys()
    if missing:
        raise AssertionError(f"dmc_attestation.json missing fields: {sorted(missing)}")
    if payload["dmc_role"] != "data_monitoring_committee_proxy":
        raise AssertionError(
            f"dmc_role must be 'data_monitoring_committee_proxy'; got {payload['dmc_role']!r}"
        )
    sha = str(payload["reviewed_08x_run_manifest_sha256"])
    if len(sha) != 64 or any(c not in "0123456789abcdef" for c in sha.lower()):
        raise AssertionError("reviewed_08x_run_manifest_sha256 must be hex sha256 (64 chars)")


def validate_08f_entry(
    *,
    dmc_attestation: dict | None,
    same_session_as_08x: bool,
    separate_session_attestation: dict | None = None,
) -> None:
    """§9.1 08F entry gate: separate Colab session by non-08X-author OR
    valid dmc_attestation.json present in 08F input directory.

    Round 7 finding #3: "separate session" must be explicitly attested via
    ``separate_session_attestation``; absence of the flag does not count as
    proof of a separate session. Acceptable inputs:

      - ``same_session_as_08x=True`` + valid ``dmc_attestation``  -> OK
      - ``same_session_as_08x=False`` + valid ``separate_session_attestation``
        + (dmc_attestation is None OR valid) -> OK
      - any other combination -> AssertionError
    """
    if dmc_attestation is not None:
        validate_dmc_attestation(dmc_attestation)
    if same_session_as_08x:
        if dmc_attestation is None:
            raise AssertionError(
                "08F entry violation: same session as 08X AND no dmc_attestation.json"
            )
        return
    # same_session_as_08x is False -> require a positive attestation file.
    if separate_session_attestation is None:
        if dmc_attestation is None:
            raise AssertionError(
                "08F entry violation: SAME_COLAB_SESSION_AS_08X=False requires "
                "separate_session_attestation.json (or a valid dmc_attestation.json "
                "as fallback); neither was provided"
            )
        return  # DMC alone is enough when same_session_as_08x is False
    validate_separate_session_attestation(separate_session_attestation)


REQUIRED_SEPARATE_SESSION_ATTESTATION_FIELDS = {
    "attestation_kind",
    "reviewer_identifier",
    "reviewed_08x_run_manifest_sha256",
    "attested_at_utc",
    "attestation_statement",
}


def validate_separate_session_attestation(payload: dict) -> None:
    """Round 7 finding #3 -- positive attestation file for the
    SAME_COLAB_SESSION_AS_08X=False branch of §9.1.

    Required fields:
      - ``attestation_kind``: must equal ``"separate_colab_session_by_non_08x_author"``
      - ``reviewer_identifier``: free-text name / stable id
      - ``reviewed_08x_run_manifest_sha256``: 64-hex sha256 of the 08X manifest
      - ``attested_at_utc``: ISO 8601 timestamp
      - ``attestation_statement``: short text
    """
    missing = REQUIRED_SEPARATE_SESSION_ATTESTATION_FIELDS - payload.keys()
    if missing:
        raise AssertionError(
            f"separate_session_attestation.json missing fields: {sorted(missing)}"
        )
    if payload["attestation_kind"] != "separate_colab_session_by_non_08x_author":
        raise AssertionError(
            "separate_session_attestation.attestation_kind must be "
            "'separate_colab_session_by_non_08x_author'; "
            f"got {payload['attestation_kind']!r}"
        )
    sha = str(payload["reviewed_08x_run_manifest_sha256"])
    if len(sha) != 64 or any(c not in "0123456789abcdef" for c in sha.lower()):
        raise AssertionError(
            "separate_session_attestation.reviewed_08x_run_manifest_sha256 "
            "must be hex sha256 (64 chars)"
        )


def validate_freeze_record(payload: dict) -> None:
    """§13.2 freeze record schema -- required fields, stage enum, scope enum,
    boolean guards (holdout closed, no official-val selection), fallback rule
    cannot reference official-validation metric (substring + normalized regex)."""
    missing = REQUIRED_FREEZE_RECORD_FIELDS - payload.keys()
    if missing:
        raise AssertionError(f"freeze record missing fields: {sorted(missing)}")
    if payload["stage"] != "08F":
        raise AssertionError(f"freeze record stage must be 08F; got {payload['stage']!r}")
    if payload["scope"] not in {"diagnostic", "validation_only"}:
        raise AssertionError(f"freeze record scope invalid: {payload['scope']!r}")
    if bool(payload["holdout_test_authorized"]):
        raise AssertionError("freeze record marks holdout_test_authorized=True (forbidden)")
    if bool(payload["official_validation_used_for_selection"]):
        raise AssertionError(
            "freeze record marks official_validation_used_for_selection=True (forbidden)"
        )
    rule_raw = str(payload["fallback_activation_rule"])
    rule_lower = rule_raw.lower()
    for tok in FORBIDDEN_FALLBACK_RULE_SUBSTRINGS:
        if tok in rule_lower:
            raise AssertionError(
                f"fallback_activation_rule references official-validation metric: {tok!r}"
            )
    rule_normalized = _normalize_fallback_rule(rule_raw)
    for pat in FORBIDDEN_FALLBACK_PATTERNS_NORMALIZED:
        m = re.search(pat, rule_normalized)
        if m is not None:
            raise AssertionError(
                f"fallback_activation_rule references official-validation metric: {m.group(0)!r}"
            )


def validate_08o_ledger_append_precedes_read(
    *,
    ledger_before_read: pd.DataFrame,
    ledger_after_read: pd.DataFrame,
) -> None:
    """§10.2 step 0 + AGENTS.md §4.3: 08O must append exactly ONE row recording
    its read intent BEFORE actually reading official validation rows. The new
    row must:
      - carry appended_by_notebook == "08O"
      - carry decision_timing == "before_official_validation_read"
      - carry official_validation_rows_inspected == 0 (the read has not happened)
      - increment the cumulative counter by exactly +1 over the prior max

    Pre-existing rows are untouched (append-only invariance: the first n_before
    rows of ledger_after_read MUST equal ledger_before_read row-for-row and
    column-for-column).
    """
    n_before = len(ledger_before_read)
    n_after = len(ledger_after_read)
    if n_after != n_before + 1:
        raise AssertionError(
            f"08O must append exactly 1 row; before={n_before}, after={n_after}"
        )
    # Append-only prefix invariance: existing rows must not be modified, dropped,
    # reordered, or have new columns inserted into them.
    if n_before > 0:
        prefix_after = ledger_after_read.iloc[:n_before].reset_index(drop=True)
        before_reset = ledger_before_read.reset_index(drop=True)
        if list(prefix_after.columns) != list(before_reset.columns):
            raise AssertionError(
                "ledger is not append-only: column set or order changed "
                f"(before={list(before_reset.columns)}, "
                f"prefix_after={list(prefix_after.columns)})"
            )
        if not prefix_after.equals(before_reset):
            raise AssertionError(
                "ledger is not append-only: pre-existing rows were modified "
                "between read and append"
            )
    new_row = ledger_after_read.iloc[-1]
    appended_by = str(new_row.get("appended_by_notebook"))
    if appended_by != "08O":
        raise AssertionError(
            f"appended_by_notebook must be '08O'; got {appended_by!r}"
        )
    timing = str(new_row.get("decision_timing"))
    if timing != "before_official_validation_read":
        raise AssertionError(
            "decision_timing must be 'before_official_validation_read'; "
            f"got {timing!r}"
        )
    inspected_at_intent = int(new_row.get("official_validation_rows_inspected", -1))
    if inspected_at_intent != 0:
        raise AssertionError(
            "official_validation_rows_inspected must be 0 at append time "
            f"(read hasn't happened yet); got {inspected_at_intent}"
        )
    last_before = (
        int(ledger_before_read["cumulative_official_validation_inspections_across_notebooks"].max())
        if not ledger_before_read.empty
        else 0
    )
    last_after = int(new_row["cumulative_official_validation_inspections_across_notebooks"])
    if last_after != last_before + 1:
        raise AssertionError(
            f"cumulative counter must be +1 from previous max; "
            f"before_max={last_before}, after_new_row={last_after}"
        )


def validate_trial_ledger_frame(df: pd.DataFrame) -> None:
    """§8.3 trial ledger schema -- required column set + compute_tier enum +
    scope/holdout/official guards + fit_status enum + failure_type enum
    (Round 7 finding #6).

    Every row must declare ``fit_status`` from ``FIT_STATUSES``. Rows with
    ``fit_status == "failed"`` must additionally carry ``failure_type`` from
    ``FAILURE_TYPES`` (a failure ledger row without a typed failure makes
    08F's failure map unauditable). Rows with ``fit_status != "failed"`` may
    leave ``failure_type`` empty.

    Raises ``AssertionError`` on contract violations.
    """
    missing = REQUIRED_TRIAL_LEDGER_COLUMNS - set(df.columns)
    if missing:
        raise AssertionError(f"08x_trial_ledger missing columns: {sorted(missing)}")
    if df.empty:
        return
    bad_tier = set(df["compute_tier"].astype(str).unique()) - set(COMPUTE_TIER_VALUES)
    if bad_tier:
        raise AssertionError(
            f"compute_tier has invalid values: {sorted(bad_tier)} "
            f"(allowed: {sorted(COMPUTE_TIER_VALUES)})"
        )
    if (df["scope"].astype(str) != "exploratory").any():
        raise AssertionError("08x_trial_ledger scope column must be 'exploratory' for all rows")
    if df["official_validation_used"].astype(bool).any():
        raise AssertionError("08x_trial_ledger contains a row with official_validation_used=True")
    if df["holdout_test_authorized"].astype(bool).any():
        raise AssertionError("08x_trial_ledger contains a row with holdout_test_authorized=True")
    bad_fit = set(df["fit_status"].astype(str).unique()) - set(FIT_STATUSES)
    if bad_fit:
        raise AssertionError(
            f"08x_trial_ledger fit_status has invalid values: {sorted(bad_fit)} "
            f"(allowed: {sorted(FIT_STATUSES)})"
        )
    failed_mask = df["fit_status"].astype(str) == "failed"
    if failed_mask.any():
        failure_values = df.loc[failed_mask, "failure_type"].astype(str)
        bad_failure = set(failure_values.unique()) - set(FAILURE_TYPES)
        if bad_failure:
            raise AssertionError(
                "08x_trial_ledger has failed rows with invalid failure_type: "
                f"{sorted(bad_failure)} (allowed: {sorted(FAILURE_TYPES)})"
            )


def validate_08x_search_space(payload: dict) -> None:
    """§7 + §11 search-space JSON guard.

    The payload MUST declare:
      - `architecture_families`: subset of SEARCH_ELIGIBLE_ARCHITECTURE_FAMILIES
      - `hpo_method`: single value from HPO_METHODS (§7.6 - same HPO per run)
      - `eligibility_thresholds.min_train_inner_lcb_delta_macro_f1`: numeric
      - `scientific_budget_cap_total_trials`: int <= TOTAL_TRIAL_BUDGET_CAP_ACROSS_ALL_FAMILIES
      - `per_family_trial_budget`: dict mapping each declared family to int
      - `official_validation_used`: must be False (08X is train-inner only; §4.1)
      - `holdout_test_authorized`: must be False (08X is train-inner only; §4.1)
    If `low_compute_mode == True`, `low_compute_submode` must be a known enum
    value and (for sub-mode B) the nested-fold protocol MUST be declared.

    Round 7 finding #4: this validator now enforces the no-official /
    no-holdout flags directly, so the contract module owns the guard rather
    than relying on the generator's defaults.
    """
    # Round 7 finding #4: contract-side guard against operator drift.
    if "official_validation_used" not in payload:
        raise AssertionError(
            "08x_search_space.official_validation_used must be declared as False"
        )
    if bool(payload["official_validation_used"]):
        raise AssertionError(
            "08x_search_space.official_validation_used=True is forbidden "
            "(08X is train-inner only per §4.1)"
        )
    if "holdout_test_authorized" not in payload:
        raise AssertionError(
            "08x_search_space.holdout_test_authorized must be declared as False"
        )
    if bool(payload["holdout_test_authorized"]):
        raise AssertionError(
            "08x_search_space.holdout_test_authorized=True is forbidden"
        )
    families = payload.get("architecture_families")
    if not isinstance(families, list) or not families:
        raise AssertionError("08x_search_space.architecture_families must be a non-empty list")
    bad_families = set(families) - set(ARCHITECTURE_FAMILIES)
    if bad_families:
        raise AssertionError(
            f"08x_search_space.architecture_families has unknown entries: {sorted(bad_families)}"
        )
    ineligible_families = set(families) - set(SEARCH_ELIGIBLE_ARCHITECTURE_FAMILIES)
    if ineligible_families:
        raise AssertionError(
            "08x_search_space.architecture_families contains families that are "
            "not 08X search-eligible until their axes are frozen in config / "
            f"search-space: {sorted(ineligible_families)}"
        )
    hpo = payload.get("hpo_method")
    if hpo not in HPO_METHODS:
        raise AssertionError(
            f"08x_search_space.hpo_method must be one of {list(HPO_METHODS)}; got {hpo!r}"
        )
    thresholds = payload.get("eligibility_thresholds", {})
    margin = thresholds.get("min_train_inner_lcb_delta_macro_f1")
    if not isinstance(margin, (int, float)):
        raise AssertionError(
            "08x_search_space.eligibility_thresholds.min_train_inner_lcb_delta_macro_f1 "
            "must be a numeric value (§9.1)"
        )
    cap = payload.get("scientific_budget_cap_total_trials")
    if not isinstance(cap, int) or cap <= 0:
        raise AssertionError(
            "08x_search_space.scientific_budget_cap_total_trials must be a positive int"
        )
    if cap > TOTAL_TRIAL_BUDGET_CAP_ACROSS_ALL_FAMILIES:
        raise AssertionError(
            f"scientific_budget_cap_total_trials={cap} exceeds §5.5 cap "
            f"{TOTAL_TRIAL_BUDGET_CAP_ACROSS_ALL_FAMILIES}"
        )
    per_family = payload.get("per_family_trial_budget", {})
    if not isinstance(per_family, dict):
        raise AssertionError("08x_search_space.per_family_trial_budget must be a dict")
    missing_family_budgets = set(families) - set(per_family.keys())
    if missing_family_budgets:
        raise AssertionError(
            "08x_search_space.per_family_trial_budget missing entries for: "
            f"{sorted(missing_family_budgets)}"
        )
    if payload.get("low_compute_mode"):
        submode = payload.get("low_compute_submode")
        if submode not in LOW_COMPUTE_SUBMODES:
            raise AssertionError(
                f"low_compute_submode must be one of {list(LOW_COMPUTE_SUBMODES)}; got {submode!r}"
            )
        if submode == "train_inner_oof_mlp_head":
            validate_low_compute_submode_b_protocol(payload)


def validate_low_compute_submode_b_protocol(search_space: dict) -> None:
    """§7.9 sub-mode B: nested-fold protocol must be fully declared.

    Raises AssertionError if any of the five required declarations is missing,
    if the outer fold scheme is not one of the allowed schemes, or if
    outer_fold_k / inner_fold_k_for_head are < 5.
    """
    missing = [f for f in LOW_COMPUTE_SUBMODE_B_REQUIRED_FIELDS if f not in search_space]
    if missing:
        raise AssertionError(
            "low_compute_submode='train_inner_oof_mlp_head' requires fields: "
            f"{missing}"
        )
    scheme = search_space["outer_fold_scheme"]
    if scheme not in LOW_COMPUTE_SUBMODE_B_ALLOWED_OUTER_FOLD_SCHEMES:
        raise AssertionError(
            f"outer_fold_scheme must be one of "
            f"{list(LOW_COMPUTE_SUBMODE_B_ALLOWED_OUTER_FOLD_SCHEMES)}; got {scheme!r}"
        )
    outer_k = search_space["outer_fold_k"]
    inner_k = search_space["inner_fold_k_for_head"]
    if not isinstance(outer_k, int) or outer_k < LOW_COMPUTE_SUBMODE_B_MIN_OUTER_FOLD_K:
        raise AssertionError(
            f"outer_fold_k must be int >= {LOW_COMPUTE_SUBMODE_B_MIN_OUTER_FOLD_K}; got {outer_k!r}"
        )
    if not isinstance(inner_k, int) or inner_k < LOW_COMPUTE_SUBMODE_B_MIN_INNER_FOLD_K:
        raise AssertionError(
            f"inner_fold_k_for_head must be int >= {LOW_COMPUTE_SUBMODE_B_MIN_INNER_FOLD_K}; "
            f"got {inner_k!r}"
        )
    expected_train = "outer_fold_i.oof_predictions_excluding_held_out_inner_fold"
    expected_eval = "outer_fold_i.oof_predictions_from_held_out_inner_fold"
    if search_space["head_train_data_source"] != expected_train:
        raise AssertionError(
            f"head_train_data_source must be {expected_train!r}; "
            f"got {search_space['head_train_data_source']!r}"
        )
    if search_space["head_eval_data_source"] != expected_eval:
        raise AssertionError(
            f"head_eval_data_source must be {expected_eval!r}; "
            f"got {search_space['head_eval_data_source']!r}"
        )


def validate_08x_run_manifest(payload: dict) -> None:
    """§13.1 08X run manifest schema."""
    missing = REQUIRED_08X_RUN_MANIFEST_FIELDS - payload.keys()
    if missing:
        raise AssertionError(f"08x_run_manifest missing fields: {sorted(missing)}")
    if payload["stage"] != "08X":
        raise AssertionError(f"08x_run_manifest stage must be '08X'; got {payload['stage']!r}")
    if payload["scope"] != "exploratory":
        raise AssertionError(
            f"08x_run_manifest scope must be 'exploratory'; got {payload['scope']!r}"
        )
    if bool(payload["official_validation_used"]):
        raise AssertionError(
            "08x_run_manifest official_validation_used=True is forbidden in 08X"
        )
    if bool(payload["holdout_test_authorized"]):
        raise AssertionError(
            "08x_run_manifest holdout_test_authorized=True is forbidden"
        )


def check_08o_real_readout_completeness(output_dir) -> dict:
    """Round 8 #1 -- strict gate on what counts as a real 08O readout.

    Returns a dict describing the per-artifact verdict so the caller can
    write it into the manifest for transparent audit:

    ::

        {
          "is_real_readout": bool,            # True only when every required
                                              # artifact passes ALL three checks
          "per_artifact": {
            "<filename>": {
              "present":         bool,        # file exists on disk
              "non_empty":       bool,        # at least one data row beyond header
              "schema_complete": bool,        # required columns are present
                                              # (extra columns are OK -- additive)
              "missing_columns": [...],       # required columns NOT present
              "row_count":       int,         # number of data rows read
            },
            ...
          },
          "missing_artifacts":  [...],        # files that flunked `present`
          "empty_artifacts":    [...],        # files that flunked `non_empty`
          "schema_drift":       [...],        # files that flunked `schema_complete`
        }

    Real-mode promotion requires EVERY required artifact in
    ``REQUIRED_08O_REAL_READOUT_ARTIFACTS`` to pass present + non_empty +
    schema_complete. Otherwise the manifest stays in stub mode.

    The previous generator-side check (`any(file non_empty)`) was too
    permissive: writing one row to one artifact would flip the manifest into
    real-readout mode (Round 8 review finding). This function returns a
    structured verdict instead of raising so the generator can record it
    on the manifest for the reviewer, but the boolean ``is_real_readout``
    must be honored by the caller.
    """
    from pathlib import Path as _Path  # local import keeps top-level surface clean

    base = _Path(output_dir)
    per_artifact: dict[str, dict] = {}
    missing_artifacts: list[str] = []
    empty_artifacts: list[str] = []
    schema_drift: list[str] = []
    for filename, required_columns in REQUIRED_08O_REAL_READOUT_ARTIFACTS.items():
        path = base / filename
        present = path.exists()
        non_empty = False
        schema_complete = False
        missing_cols: list[str] = []
        row_count = 0
        if present:
            try:
                df = pd.read_csv(path)
            except (pd.errors.EmptyDataError, pd.errors.ParserError):
                # File exists but is unreadable -- treat as not non_empty and
                # not schema_complete so it stays in stub mode.
                df = None
            if df is not None:
                row_count = int(len(df))
                non_empty = row_count > 0
                missing_cols = sorted(set(required_columns) - set(df.columns))
                schema_complete = not missing_cols
        verdict = {
            "present": present,
            "non_empty": non_empty,
            "schema_complete": schema_complete,
            "missing_columns": missing_cols,
            "row_count": row_count,
        }
        per_artifact[filename] = verdict
        if not present:
            missing_artifacts.append(filename)
        if present and not non_empty:
            empty_artifacts.append(filename)
        if present and non_empty and not schema_complete:
            schema_drift.append(filename)
    is_real = (
        not missing_artifacts
        and not empty_artifacts
        and not schema_drift
    )
    return {
        "is_real_readout": is_real,
        "per_artifact": per_artifact,
        "missing_artifacts": missing_artifacts,
        "empty_artifacts": empty_artifacts,
        "schema_drift": schema_drift,
    }


def validate_08o_run_manifest(payload: dict) -> None:
    """§13.3 08O run manifest schema -- required fields, scope guard,
    same-row dummy / per-ticker / seed summary presence flags.

    Round 7 finding #1: the manifest now distinguishes a schema-only stub
    (no real official-validation read happened; the artifact rows have only
    headers) from a true readout. The optional ``schema_only_stub`` field
    governs this:

      - ``schema_only_stub=True``  -> ``*_present`` may be False;
        ``allowed_wording_bucket`` MUST be ``"no_candidate_freezable"`` so
        downstream consumers cannot mistake stubs for evidence.
      - ``schema_only_stub=False`` (or absent) -> the legacy contract:
        all three ``*_present`` flags MUST be True before the manifest is
        considered a real readout.
    """
    missing = REQUIRED_08O_RUN_MANIFEST_FIELDS - payload.keys()
    if missing:
        raise AssertionError(f"08o_run_manifest missing fields: {sorted(missing)}")
    if payload["stage"] != "08O":
        raise AssertionError(f"08o_run_manifest stage must be '08O'; got {payload['stage']!r}")
    if payload["scope"] != "validation_only":
        raise AssertionError(
            f"08o_run_manifest scope must be 'validation_only'; got {payload['scope']!r}"
        )
    if bool(payload["official_validation_used_for_selection"]):
        raise AssertionError(
            "08o_run_manifest official_validation_used_for_selection=True is forbidden"
        )
    if bool(payload["holdout_test_authorized"]):
        raise AssertionError(
            "08o_run_manifest holdout_test_authorized=True is forbidden"
        )
    schema_only_stub = bool(payload.get("schema_only_stub", False))
    if schema_only_stub:
        # Stub mode: stub manifests MUST carry the no-candidate wording bucket
        # so a downstream paper / static gate cannot accidentally treat empty
        # artifacts as evidence (Round 7 finding #1).
        if payload["allowed_wording_bucket"] != "no_candidate_freezable":
            raise AssertionError(
                "08o_run_manifest schema_only_stub=True requires "
                "allowed_wording_bucket='no_candidate_freezable'; "
                f"got {payload['allowed_wording_bucket']!r}"
            )
    else:
        # Real-readout mode: presence flags must be True.
        for flag in (
            "same_row_dummy_present", "per_ticker_present", "seed_summary_present"
        ):
            if not bool(payload[flag]):
                raise AssertionError(
                    f"08o_run_manifest {flag}=False; required artifacts missing per §13.3"
                )
    if payload["allowed_wording_bucket"] not in ALLOWED_WORDING_BUCKETS:
        raise AssertionError(
            f"08o_run_manifest allowed_wording_bucket must be one of "
            f"{list(ALLOWED_WORDING_BUCKETS)}; got {payload['allowed_wording_bucket']!r}"
        )

## Config And Run Switches

Scope, output dir, 13 RUN_08X_*/F_*/O_* research switches, the optional package-backed stage boundary (all default `False`), operator acknowledgements, locked Stage 0 candidate, design-doc + operator-readout SHA pins, and output file paths.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import re
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# Scope -- bind every artifact and ledger row to validation_only / exploratory.
NOTEBOOK08_SCOPE = "validation_only"
NOTEBOOK08_VERSION = "2026-06-06-mvp"

# Input directories. 08X reads ONLY the official training partition and N05
# decision record (for the frozen Stage 0 anchor). 08F reads ONLY 08X
# artifacts + N07 ledger. 08O reads the official-validation partition exactly
# ONCE for the frozen candidate.
NOTEBOOK05_RESULTS_DIR = Path("/content/notebook05_lightgbm_tuning_results")
NOTEBOOK06_RESULTS_DIR = Path("/content/notebook06_selective_no_trade_calibration_results")
NOTEBOOK07_RESULTS_DIR = Path("/content/notebook07_validation_synthesis_and_gap_audit_results")
OUTPUT_DIR = Path("/content/notebook08_deep_sequence_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Project-level validation-budget ledger -- single source of truth per
# AGENTS.md §4.3. 08O appends BEFORE reading any official-validation metric.
PROJECT_VALIDATION_BUDGET_LEDGER_PATH = (
    NOTEBOOK07_RESULTS_DIR / "notebook07_validation_budget_ledger.csv"
)

# Run switches -- design §12. Every gate defaults False; nothing runs unless
# the operator explicitly flips a switch in a copy of the notebook.
RUN_08X_SCHEMA_SMOKE = False
RUN_08X_BUILD_TRAIN_INNER_FOLDS = False
RUN_08X_SEARCH_SPACE_DRY_RUN = False
RUN_08X_QUICK_SEARCH = False
RUN_08X_MEDIUM_SEARCH = False
RUN_08X_AGGRESSIVE_SEARCH = False
RUN_08X_AGGREGATE_FAILURE_MAP = False

RUN_08F_CONTRACT_GATE = False
RUN_08F_CANDIDATE_COMPRESSION = False
RUN_08F_WRITE_FREEZE_RECORD = False

RUN_08O_ENTRY_GATE = False
RUN_08O_OFFICIAL_VALIDATION_READOUT = False
RUN_08O_AGGREGATE_AND_WRITE_MANIFEST = False

RUN_PACKAGE_BACKED_STAGE = False
PACKAGE_INSTALL_ENABLED = False
PACKAGE_REPO_URL = "https://github.com/zkc768/intraday_stock_direction_research.git"
PACKAGE_GIT_COMMIT = "0000000000000000000000000000000000000000"
PACKAGE_STAGE_NAME = "deep_sequence_exploration"
PACKAGE_OFFICIAL_VALIDATION_PREDICTIONS_CSV = (
    OUTPUT_DIR / "official_validation_predictions.csv"
)

BACKUP_NOTEBOOK08_TO_GOOGLE_DRIVE = False
DRIVE_BACKUP_FOLDER_ID = ""
DRIVE_BACKUP_PREFIX = "notebook08_deep_sequence"

# Operator acknowledgements -- design §16 "must not" list. Each ack defaults
# False so a Colab fork that flips only RUN_* still trips the entry gate.
OPERATOR_ACKNOWLEDGES_08X_IS_EXPLORATORY_ONLY = False
OPERATOR_ACKNOWLEDGES_NO_HOLDOUT_TEST = False
OPERATOR_ACKNOWLEDGES_NO_OFFICIAL_VAL_SELECTION = False
OPERATOR_ACKNOWLEDGES_NO_THRESHOLD_FISHING = False
OPERATOR_ACKNOWLEDGES_FALLBACK_RULE_FROZEN_BEFORE_INSPECTION = False

# Round 7 finding #3 -- safe-by-default: assume 08F runs in the SAME Colab
# session as 08X. The operator must EITHER drop a valid dmc_attestation.json
# in OUTPUT_DIR (and keep this flag True), OR flip this to False AND drop a
# valid separate_session_attestation.json in OUTPUT_DIR (positive proof of a
# separate session by a non-08X-author). An absent flag is NOT proof of a
# separate session; see ``validate_08f_entry`` in the canonical contract module.
SAME_COLAB_SESSION_AS_08X = True
SEPARATE_SESSION_ATTESTATION_PATH = OUTPUT_DIR / "separate_session_attestation.json"
DMC_ATTESTATION_PATH = OUTPUT_DIR / "dmc_attestation.json"

# Design doc pin per §10.1 entry gate. Set at freeze time, verified by 08O.
DESIGN_DOC_PATH = (
    "docs/NOTEBOOK08_DEEP_SEQUENCE_EXPLORATION_FREEZE_READOUT_TECHNICAL_DESIGN_2026-06-06.md"
)
EXPECTED_DESIGN_DOC_SHA256 = ""  # 64-hex sha; pinned by operator before 08O
DESIGN_DOC_SHA256_OBSERVED = ""

# §10.1 OPERATOR_READOUT_AUTHORIZATION_SHA inputs -- fixed order matters.
# Recomputed by 08O entry gate using ``operator_readout_authorization_sha``
# and compared against ``EXPECTED_OPERATOR_READOUT_AUTHORIZATION_SHA``.
EXPECTED_OPERATOR_READOUT_AUTHORIZATION_SHA = ""
OPERATOR_READOUT_AUTHORIZATION_SHA_OBSERVED = ""

# Stage 0 locked candidate tuple per design §1.1 -- the only label/feature/
# window combination 08 is authorized to explore.
LOCKED_CANDIDATE_TUPLE = {
    "label_config": "h03_bps1p5",
    "feature_set": "price_volume_time",
    "window_size": 20,
}

# Architecture-family budget cap (§5.5) and tier escalation gate (§11.1).
TOTAL_TRIAL_BUDGET_CAP_DEFAULT = TOTAL_TRIAL_BUDGET_CAP_ACROSS_ALL_FAMILIES

# §7.9 low-compute submode toggles. Default is OFF; operator must declare in
# 08x_search_space.json before enabling. Sub-mode B requires the nested-fold
# protocol below to be fully declared.
LOW_COMPUTE_MODE = False
LOW_COMPUTE_SUBMODE = ""  # "deterministic_agg" | "train_inner_oof_mlp_head"

# Frozen seed list. Mirrors N05/N06; operator can extend but only by editing
# 08x_search_space.json (which then gets sha256-stamped).
DEFAULT_SEED_LIST = (260501, 260502, 260503)

# Default per-family trial budget -- replicated into 08x_search_space.json by
# RUN_08X_SEARCH_SPACE_DRY_RUN; lower than design §11 "aggressive" caps so the
# MVP smoke test stays cheap. Real exploration overrides via search-space.
DEFAULT_PER_FAMILY_BUDGET = {family: 5 for family in SEARCH_ELIGIBLE_ARCHITECTURE_FAMILIES}

# Output artifact filenames -- single source of truth so the contract module
# and the runtime cells agree byte-for-byte (per design §13.1 / §13.2 / §13.3).
OUTPUT_FILES = {
    # 08X
    "08x_search_space": OUTPUT_DIR / "08x_search_space.json",
    "08x_train_inner_folds": OUTPUT_DIR / "08x_train_inner_folds.csv",
    "08x_trial_ledger": OUTPUT_DIR / "08x_trial_ledger.csv",
    "08x_fold_results": OUTPUT_DIR / "08x_fold_results.csv",
    "08x_seed_summary": OUTPUT_DIR / "08x_seed_summary.csv",
    "08x_failure_ledger": OUTPUT_DIR / "08x_failure_ledger.csv",
    "08x_candidate_compression_table": OUTPUT_DIR / "08x_candidate_compression_table.csv",
    "08x_run_manifest": OUTPUT_DIR / "08x_run_manifest.json",
    "08x_environment_manifest": OUTPUT_DIR / "08x_environment_manifest.json",
    "08x_tier_escalation_blocked": OUTPUT_DIR / "08x_tier_escalation_blocked.json",
    # 08F
    "08f_candidate_freeze_record_json": OUTPUT_DIR / "08f_candidate_freeze_record.json",
    "08f_candidate_freeze_record_md": OUTPUT_DIR / "08f_candidate_freeze_record.md",
    "08f_candidate_compression_audit": OUTPUT_DIR / "08f_candidate_compression_audit.csv",
    "08f_static_gate_report": OUTPUT_DIR / "08f_static_gate_report.json",
    "08f_no_candidate_freezable": OUTPUT_DIR / "08f_no_candidate_freezable.json",
    # 08O
    "08o_validation_readout": OUTPUT_DIR / "08o_validation_readout.csv",
    "08o_validation_per_ticker": OUTPUT_DIR / "08o_validation_per_ticker.csv",
    "08o_seed_summary": OUTPUT_DIR / "08o_seed_summary.csv",
    "08o_same_row_baselines": OUTPUT_DIR / "08o_same_row_baselines.csv",
    "08o_concentration_guardrails": OUTPUT_DIR / "08o_concentration_guardrails.csv",
    "08o_failure_rows": OUTPUT_DIR / "08o_failure_rows.csv",
    "08o_decision_record": OUTPUT_DIR / "08o_decision_record.json",
    "08o_run_manifest": OUTPUT_DIR / "08o_run_manifest.json",
    # Misc
    "package_stage_invocation_manifest": OUTPUT_DIR
    / "notebook08_package_stage_invocation_manifest.json",
    "drive_backup_manifest": OUTPUT_DIR / "notebook08_drive_backup_manifest.json",
}

RUN_SWITCHES = {
    "RUN_08X_SCHEMA_SMOKE": RUN_08X_SCHEMA_SMOKE,
    "RUN_08X_BUILD_TRAIN_INNER_FOLDS": RUN_08X_BUILD_TRAIN_INNER_FOLDS,
    "RUN_08X_SEARCH_SPACE_DRY_RUN": RUN_08X_SEARCH_SPACE_DRY_RUN,
    "RUN_08X_QUICK_SEARCH": RUN_08X_QUICK_SEARCH,
    "RUN_08X_MEDIUM_SEARCH": RUN_08X_MEDIUM_SEARCH,
    "RUN_08X_AGGRESSIVE_SEARCH": RUN_08X_AGGRESSIVE_SEARCH,
    "RUN_08X_AGGREGATE_FAILURE_MAP": RUN_08X_AGGREGATE_FAILURE_MAP,
    "RUN_08F_CONTRACT_GATE": RUN_08F_CONTRACT_GATE,
    "RUN_08F_CANDIDATE_COMPRESSION": RUN_08F_CANDIDATE_COMPRESSION,
    "RUN_08F_WRITE_FREEZE_RECORD": RUN_08F_WRITE_FREEZE_RECORD,
    "RUN_08O_ENTRY_GATE": RUN_08O_ENTRY_GATE,
    "RUN_08O_OFFICIAL_VALIDATION_READOUT": RUN_08O_OFFICIAL_VALIDATION_READOUT,
    "RUN_08O_AGGREGATE_AND_WRITE_MANIFEST": RUN_08O_AGGREGATE_AND_WRITE_MANIFEST,
}

# Pre-registration constants table mirror -- sourced from the canonical contract module.
PRE_REGISTRATION_CONSTANTS = {
    "improvement_threshold_delta_macro_f1_lcb_95": IMPROVEMENT_THRESHOLD_DELTA_MACRO_F1_LCB_95,
    "improvement_threshold_positive_ticker_count_min": IMPROVEMENT_THRESHOLD_POSITIVE_TICKER_COUNT_MIN,
    "fusion_min_lcb_advantage_over_components": FUSION_MIN_LCB_ADVANTAGE_OVER_COMPONENTS,
    "candidate_eligibility_min_train_inner_lcb_delta": CANDIDATE_ELIGIBILITY_MIN_TRAIN_INNER_LCB_DELTA,
    "paper_safe_score_weight_lcb_delta": PAPER_SAFE_SCORE_WEIGHT_LCB_DELTA,
    "paper_safe_score_weight_mean_delta": PAPER_SAFE_SCORE_WEIGHT_MEAN_DELTA,
    "paper_safe_score_weight_seed_stability": PAPER_SAFE_SCORE_WEIGHT_SEED_STABILITY,
    "paper_safe_score_weight_fold_consistency": PAPER_SAFE_SCORE_WEIGHT_FOLD_CONSISTENCY,
    "paper_safe_score_weight_per_ticker": PAPER_SAFE_SCORE_WEIGHT_PER_TICKER,
    "paper_safe_score_penalty_complexity": PAPER_SAFE_SCORE_PENALTY_COMPLEXITY,
    "paper_safe_score_penalty_compute": PAPER_SAFE_SCORE_PENALTY_COMPUTE,
    "class_collapse_pred_rate_min": CLASS_COLLAPSE_PRED_RATE_MIN,
    "total_trial_budget_cap_across_all_families": TOTAL_TRIAL_BUDGET_CAP_ACROSS_ALL_FAMILIES,
}

print("Notebook 08 scope:", NOTEBOOK08_SCOPE)
print("Notebook 08 version:", NOTEBOOK08_VERSION)
print("Output dir:", OUTPUT_DIR)
print("Run switches:", RUN_SWITCHES)

## Optional Package Install Boundary

Default inert. Package install is allowed only after the operator replaces the placeholder `PACKAGE_GIT_COMMIT` with an exact 40-hex commit and explicitly enables `PACKAGE_INSTALL_ENABLED`.

In [ ]:
if PACKAGE_INSTALL_ENABLED:
    if not re.fullmatch(r"[0-9a-f]{40}", PACKAGE_GIT_COMMIT):
        raise ValueError(
            "PACKAGE_GIT_COMMIT must be a 40-character lowercase hex commit"
        )
    if PACKAGE_GIT_COMMIT == "0" * 40:
        raise ValueError(
            "PACKAGE_GIT_COMMIT is still the inert placeholder; replace it "
            "with the exact commit before enabling package install"
        )
    if PACKAGE_REPO_URL.endswith("/"):
        raise ValueError("PACKAGE_REPO_URL must not end with a slash")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            f"git+{PACKAGE_REPO_URL}@{PACKAGE_GIT_COMMIT}",
        ]
    )
else:
    print("[package] PACKAGE_INSTALL_ENABLED = False (no install)")

## Runtime Helpers

Canonical JSON/CSV writers, sha256 / env / git helpers, ledger append-before-read guard, `z_in_tier` z-score, trial-row envelope, and the `OPERATOR_READOUT_AUTHORIZATION_SHA` runtime hasher per §10.1.

In [ ]:
def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()


def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        json.dump(payload, handle, indent=2, sort_keys=True)


def sha256_file(path):
    hasher = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()


def canonical_csv_bytes(df):
    return df.to_csv(index=False, lineterminator="\n").encode("utf-8")


def canonical_json_bytes(payload):
    return json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")


def python_env_sha256():
    """Sha256 of sorted ``pip freeze`` output for §13.2
    ``frozen_python_env_hash``."""
    try:
        out = subprocess.check_output(
            [sys.executable, "-m", "pip", "freeze"],
            stderr=subprocess.STDOUT,
            text=True,
        )
    except (subprocess.CalledProcessError, OSError) as err:
        return f"pip_freeze_failed:{err}"
    lines = sorted(line.strip() for line in out.splitlines() if line.strip())
    return sha256_bytes("\n".join(lines).encode("utf-8"))


def git_head_sha():
    """Return full sha of current HEAD or 'no_git'/'dirty' marker."""
    try:
        head = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.STDOUT, text=True
        ).strip()
    except (subprocess.CalledProcessError, OSError):
        return "no_git"
    try:
        dirty = subprocess.check_output(
            ["git", "status", "--porcelain"], stderr=subprocess.STDOUT, text=True
        ).strip()
    except (subprocess.CalledProcessError, OSError):
        dirty = ""
    if dirty:
        # Per design §13.2 -- uncommitted changes block freeze. 08F entry gate
        # will refuse a freeze record carrying a "-dirty" sha.
        return f"{head}-dirty"
    return head


def read_ledger_or_empty(path):
    """Return on-disk validation_budget_ledger as a DataFrame, or an empty
    one with the §07C column set so append-only invariance still holds."""
    columns = [
        "artifact",
        "notebook_stage",
        "decision_made",
        "decision_timing",
        "decision_surface",
        "model_families_considered",
        "profiles_or_trials_considered",
        "seeds_used",
        "thresholds_or_coverages_considered",
        "official_validation_rows_inspected",
        "cumulative_official_validation_inspections_across_notebooks",
        "train_inner_only_decision",
        "official_validation_informed_decision",
        "diagnostic_only_readout",
        "holdout_test_contact",
        "allowed_wording",
        "forbidden_wording",
        "risk_note",
        "appended_by_notebook",
        "appended_at_utc",
    ]
    if not Path(path).exists():
        return pd.DataFrame(columns=columns)
    df = pd.read_csv(path)
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise AssertionError(
            f"validation_budget_ledger missing columns: {missing} (read from {path})"
        )
    return df[columns]


def append_ledger_row(existing_df, new_row_dict):
    """Append a single row to the ledger; return (new_df, validation_callable).

    The caller invokes ``validate_08o_ledger_append_precedes_read`` with
    (existing_df, new_df) BEFORE actually reading official validation, per
    AGENTS.md §4.3 and design §10.2 step 0.
    """
    new_df = pd.concat([existing_df, pd.DataFrame([new_row_dict])], ignore_index=True)
    return new_df


def write_ledger(path, df):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, lineterminator="\n")


def low_compute_z(values):
    """Z-score normalization with an edge-case-aware std guard. Returns
    ``np.zeros_like`` when ``len(values) < 2`` or ``std == 0`` so single-tier
    trials contribute 0 to a penalty term (§9.2 edge cases)."""
    arr = np.asarray(values, dtype=float)
    if arr.size < 2:
        return np.zeros_like(arr)
    std = arr.std(ddof=0)
    if std == 0.0:
        return np.zeros_like(arr)
    return (arr - arr.mean()) / std


def lcb_95(values):
    """Lower 95% confidence bound = mean - 1.96 * std / sqrt(n). Returns nan
    on len < 2."""
    arr = np.asarray(values, dtype=float)
    if arr.size < 2:
        return float("nan")
    return float(arr.mean() - 1.96 * arr.std(ddof=1) / math.sqrt(arr.size))


def operator_readout_authorization_sha_runtime(fixed_order_inputs):
    """Inline version of canonical contract operator_readout_authorization_sha
    using the symbols available in the notebook namespace. Kept here so the
    notebook can compute the §10.1 SHA without re-importing the contract
    module (the contract module is already inlined as cell 2)."""
    hasher = hashlib.sha256()
    for path, mode in fixed_order_inputs:
        rel_path = str(path).replace("\\", "/")
        path_bytes = rel_path.encode("utf-8")
        hasher.update(len(path_bytes).to_bytes(8, "big"))
        hasher.update(path_bytes)
        raw_bytes = Path(path).read_bytes()
        if mode == "json_canonical":
            obj = json.loads(raw_bytes.decode("utf-8"))
            canonical = json.dumps(
                obj,
                sort_keys=True,
                separators=(",", ":"),
                ensure_ascii=False,
                allow_nan=False,
            ).encode("utf-8")
        elif mode == "text_lf":
            canonical = raw_bytes.replace(b"\r\n", b"\n")
        else:
            raise ValueError(
                f"unsupported canonicalization mode: {mode!r}"
            )
        hasher.update(len(canonical).to_bytes(8, "big"))
        hasher.update(canonical)
    return hasher.hexdigest()


# ---------- Trial result envelope ------------------------------------------


def make_trial_row(*, trial_id, candidate_family, candidate_id, config_hash,
                   fold_id, seed, budget_tier, compute_tier="full_compute"):
    """Build a row matching the §8.3 trial-ledger schema with all metric / fit
    fields defaulted to NaN. The trial-loop writes back into the dict and the
    schema validator runs at write time."""
    return {
        "trial_id": trial_id,
        "candidate_family": candidate_family,
        "candidate_id": candidate_id,
        "config_hash": config_hash,
        "fold_id": fold_id,
        "seed": seed,
        "budget_tier": budget_tier,
        "max_epochs": 0,
        "actual_epochs": 0,
        "early_stop_reason": "",
        "fit_status": "pending",
        "failure_type": "",
        "failure_message": "",
        "train_inner_fit_n": 0,
        "train_inner_validation_n": 0,
        "macro_f1": float("nan"),
        "balanced_accuracy": float("nan"),
        "accuracy": float("nan"),
        "stratified_dummy_macro_f1_same_rows": float("nan"),
        "delta_macro_f1_vs_dummy": float("nan"),
        "class0_pred_rate": float("nan"),
        "class1_pred_rate": float("nan"),
        "ticker_max_share": float("nan"),
        "actual_wall_clock_seconds": 0.0,
        "peak_memory_mb": 0.0,
        "gpu_seconds_or_null": None,
        "compute_tier": compute_tier,
        "scope": "exploratory",
        "official_validation_used": False,
        "holdout_test_authorized": False,
    }

## Optional Package-Backed Stage Invocation

Default inert. When enabled after exact-commit package install, invokes `intraday_research.stages.deep_sequence_exploration.run_stage` with the same research switches and records a package invocation manifest. Inline notebook cells remain available for review.

In [ ]:
if RUN_PACKAGE_BACKED_STAGE:
    if not PACKAGE_INSTALL_ENABLED:
        raise ValueError(
            "RUN_PACKAGE_BACKED_STAGE requires PACKAGE_INSTALL_ENABLED=True"
        )
    if PACKAGE_GIT_COMMIT == "0" * 40:
        raise ValueError(
            "RUN_PACKAGE_BACKED_STAGE requires a real exact git commit pin"
        )

    from intraday_research.stages.deep_sequence_exploration import (
        run_stage as run_deep_sequence_exploration_stage,
    )

    package_run_switches = {
        name: value
        for name, value in RUN_SWITCHES.items()
        if name.startswith("RUN_08")
    }
    package_config = {
        "stage_name": PACKAGE_STAGE_NAME,
        "validation_scope": NOTEBOOK08_SCOPE,
        "holdout_test_contact": False,
        "run_switches": package_run_switches,
        "outputs": {"results_dir": str(OUTPUT_DIR)},
        "inputs": {
            "official_validation_predictions_csv": str(
                PACKAGE_OFFICIAL_VALIDATION_PREDICTIONS_CSV
            ),
            "08o_decision_record": str(OUTPUT_FILES["08o_decision_record"]),
            "08f_candidate_freeze_record": str(
                OUTPUT_FILES["08f_candidate_freeze_record_json"]
            ),
            "validation_budget_ledger": str(PROJECT_VALIDATION_BUDGET_LEDGER_PATH),
        },
        "policy": {
            "validation_budget_ledger": {
                "path": str(PROJECT_VALIDATION_BUDGET_LEDGER_PATH)
            },
            "official_validation": {
                "expected_tickers": ["CSCO", "JPM", "KO", "MSFT", "WMT"],
                "holdout_test_contact": False,
            },
            "package_install": {
                "repo_url": PACKAGE_REPO_URL,
                "git_commit": PACKAGE_GIT_COMMIT,
            },
        },
        "constants": PRE_REGISTRATION_CONSTANTS,
    }
    run_deep_sequence_exploration_stage(package_config, output_dir=OUTPUT_DIR)
    package_invocation_manifest = {
        "stage_name": PACKAGE_STAGE_NAME,
        "package_repo_url": PACKAGE_REPO_URL,
        "package_git_commit": PACKAGE_GIT_COMMIT,
        "run_switches": package_run_switches,
        "validation_scope": NOTEBOOK08_SCOPE,
        "holdout_test_contact": False,
        "invocation_status": "completed",
        "written_at_utc": utc_now_iso(),
    }
    write_json(
        OUTPUT_FILES["package_stage_invocation_manifest"],
        package_invocation_manifest,
    )
else:
    print("[package] RUN_PACKAGE_BACKED_STAGE = False (no package stage invocation)")

## 08X - Schema Smoke

Verifies the output directory and the project-level validation-budget ledger path. No model fits.

In [ ]:
if RUN_08X_SCHEMA_SMOKE:
    print("[08X] schema smoke -- validating output directory + ledger seed")
    # Verify OUTPUT_DIR is writeable and the project-level validation-budget
    # ledger schema is reachable (08O will append to it).
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    smoke_payload = {
        "stage": "08X",
        "scope": NOTEBOOK08_SCOPE,
        "smoke_at_utc": utc_now_iso(),
        "output_dir": str(OUTPUT_DIR),
        "project_ledger_path": str(PROJECT_VALIDATION_BUDGET_LEDGER_PATH),
        "project_ledger_present": PROJECT_VALIDATION_BUDGET_LEDGER_PATH.exists(),
        "locked_candidate_tuple": LOCKED_CANDIDATE_TUPLE,
        "pre_registration_constants": PRE_REGISTRATION_CONSTANTS,
        "run_switches": RUN_SWITCHES,
        "official_validation_used": False,
        "holdout_test_authorized": False,
    }
    write_json(OUTPUT_DIR / "08x_schema_smoke.json", smoke_payload)
    print("[08X] schema smoke OK ->", OUTPUT_DIR / "08x_schema_smoke.json")
else:
    print("[08X] RUN_08X_SCHEMA_SMOKE = False (no work)")

## 08X - Build Train-Inner Folds

Writes the fold spec per §8.2 (per-ticker chronological, embargoed, no-cross-day, no-cross-ticker, train-inner-fit-only preprocessing) and an empty fold result CSV. No fold rows are scored until the trial loop runs.

In [ ]:
if RUN_08X_BUILD_TRAIN_INNER_FOLDS:
    print("[08X] building train-inner folds per design §8.2")
    # Per design §8.2: per-ticker chronological split first, then pool.
    # The actual fold construction depends on the locked Stage 0 data layout
    # (raw bars in /content/stage0_raw_stock_data) and is implemented when
    # the operator runs this cell against real bars. Until then, this writes
    # a fold spec describing the protocol that MUST be honored.
    fold_spec = {
        "stage": "08X",
        "scope": NOTEBOOK08_SCOPE,
        "fold_policy": "embargoed_train_inner_folds",
        "purge_horizon_bars": 0,  # operator sets per LOCKED_CANDIDATE_TUPLE.horizon
        "embargo_bars": 1,         # one-bar embargo at fold boundaries
        "per_ticker_chronological_first": True,
        "no_window_crosses_trading_day": True,
        "no_window_crosses_ticker": True,
        "preprocessing_fit_on_train_inner_fit_only": True,
        "outer_official_train_partition_only": True,
        "official_validation_partition_read": False,
        "closed_holdout_test_read": False,
        "low_compute_mode": LOW_COMPUTE_MODE,
        "low_compute_submode": LOW_COMPUTE_SUBMODE,
        "built_at_utc": utc_now_iso(),
    }
    write_json(OUTPUT_DIR / "08x_train_inner_folds.json", fold_spec)
    # Emit an empty CSV with the fold-result schema so downstream cells can
    # append rows without schema drift.
    fold_columns = [
        "fold_id", "fold_kind", "train_inner_fit_start_utc",
        "train_inner_fit_end_utc", "train_inner_validation_start_utc",
        "train_inner_validation_end_utc", "fold_n", "purge_n", "embargo_n",
    ]
    pd.DataFrame(columns=fold_columns).to_csv(
        OUTPUT_FILES["08x_train_inner_folds"], index=False, lineterminator="\n"
    )
    print("[08X] train-inner fold spec ->", OUTPUT_DIR / "08x_train_inner_folds.json")
else:
    print("[08X] RUN_08X_BUILD_TRAIN_INNER_FOLDS = False (no work)")

## 08X - Search Space Dry Run

Emits `08x_search_space.json` with the MVP families, per-family budget, single HPO method (§7.6), eligibility margin (§9.1), and scientific budget cap (§5.5). Calls `validate_08x_search_space` for schema sign-off.

In [ ]:
if RUN_08X_SEARCH_SPACE_DRY_RUN:
    print("[08X] search-space dry run -- writing 08x_search_space.json")
    # Construct a minimal compliant search space (§7 + §11). The contract's
    # validator enforces: families subset of SEARCH_ELIGIBLE_ARCHITECTURE_FAMILIES,
    # single hpo_method, numeric eligibility margin, scientific budget cap <= §5.5
    # cap, per_family_trial_budget for every declared family. The MVP includes
    # last_step_lightgbm_control + a small deep slice so the failure ledger
    # exercises the not_implemented path.
    families_for_mvp = [
        "last_step_lightgbm_control",
        "ms_dlinear_tcn",
        "dlinear_only",
        "tcn_only",
    ]
    search_space = {
        "search_space_version": NOTEBOOK08_VERSION,
        "stage": "08X",
        "scope": NOTEBOOK08_SCOPE,
        "architecture_families": families_for_mvp,
        "per_family_trial_budget": {
            family: int(DEFAULT_PER_FAMILY_BUDGET.get(family, 5))
            for family in families_for_mvp
        },
        "hpo_method": "random_search",
        "eligibility_thresholds": {
            "min_train_inner_lcb_delta_macro_f1": (
                CANDIDATE_ELIGIBILITY_MIN_TRAIN_INNER_LCB_DELTA
            ),
        },
        "scientific_budget_cap_total_trials": int(
            TOTAL_TRIAL_BUDGET_CAP_DEFAULT
        ),
        "fusion_min_lcb_advantage_over_components": (
            FUSION_MIN_LCB_ADVANTAGE_OVER_COMPONENTS
        ),
        "low_compute_mode": LOW_COMPUTE_MODE,
        "low_compute_submode": LOW_COMPUTE_SUBMODE,
        "seed_list": list(DEFAULT_SEED_LIST),
        "official_validation_used": False,
        "holdout_test_authorized": False,
        "stamped_at_utc": utc_now_iso(),
    }
    validate_08x_search_space(search_space)
    write_json(OUTPUT_FILES["08x_search_space"], search_space)
    sha = sha256_file(OUTPUT_FILES["08x_search_space"])
    print("[08X] search space sha256 =", sha)
    print("[08X] search space ->", OUTPUT_FILES["08x_search_space"])
else:
    print("[08X] RUN_08X_SEARCH_SPACE_DRY_RUN = False (no work)")

## 08X - Quick Search

Minimal-budget trial loop (4-8 configs, 1-2 folds, 1-2 seeds). `last_step_lightgbm_control` is real; deep families emit `failure_type="not_implemented"` rows so the ledger / failure-map paths are exercised.

In [ ]:
def _run_one_trial(*, family, config_hash, fold_id, seed, budget_tier, compute_tier):
    """Execute one trial. ``last_step_lightgbm_control`` runs a real LightGBM
    classifier; deep families raise NotImplementedError and the calling loop
    converts that into a well-formed failure row."""
    row = make_trial_row(
        trial_id=f"{family}::{config_hash}::fold{fold_id}::seed{seed}",
        candidate_family=family,
        candidate_id=f"{family}_{config_hash[:8]}",
        config_hash=config_hash,
        fold_id=fold_id,
        seed=seed,
        budget_tier=budget_tier,
        compute_tier=compute_tier,
    )
    if family == "last_step_lightgbm_control":
        # LightGBM "last step" control: MVP emits a pending row; operator
        # wires the real fit against the train-inner fold's tabular features
        # (extracted from the LOCKED_CANDIDATE_TUPLE last bar of each window)
        # in a follow-up PR. Until then this row is NOT evidence; static gate
        # treats fit_status="pending_last_step_lightgbm" as pre-fit per
        # FIT_STATUSES enum.
        row["fit_status"] = "pending_last_step_lightgbm"
        row["failure_type"] = ""
        row["failure_message"] = (
            "MVP stub: operator must wire LightGBM call to fold rows "
            "before this trial counts toward paper_safe_score"
        )
    else:
        # Deep families (DLinear / TCN / GRU / LSTM / fusion) -- MVP stubs.
        raise NotImplementedError(
            f"deep-sequence family {family!r} not implemented in MVP "
            f"(design §7.2/§7.3); deferred to follow-up PR"
        )
    return row


def _run_trial_loop(*, budget_tier, search_space):
    """Iterate over (family, config, fold, seed) per search space; write rows
    to the trial ledger + failure ledger; honor the scientific budget cap."""
    rows = []
    failure_rows = []
    cap = int(search_space["scientific_budget_cap_total_trials"])
    total = 0
    families = list(search_space["architecture_families"])
    seeds = list(search_space["seed_list"])
    for family in families:
        per_family_budget = int(search_space["per_family_trial_budget"][family])
        for config_index in range(per_family_budget):
            for fold_id in range(min(3, per_family_budget)):
                for seed in seeds:
                    if total >= cap:
                        print("[08X] hit scientific_budget_cap_total_trials",
                              cap, "-- stopping early")
                        return rows, failure_rows
                    total += 1
                    config_hash = sha256_bytes(
                        f"{family}::{config_index}::{budget_tier}".encode("utf-8")
                    )
                    try:
                        row = _run_one_trial(
                            family=family,
                            config_hash=config_hash,
                            fold_id=fold_id,
                            seed=seed,
                            budget_tier=budget_tier,
                            compute_tier="full_compute",
                        )
                        rows.append(row)
                    except NotImplementedError as err:
                        failure_row = make_trial_row(
                            trial_id=(
                                f"{family}::{config_hash}::fold{fold_id}::"
                                f"seed{seed}"
                            ),
                            candidate_family=family,
                            candidate_id=f"{family}_{config_hash[:8]}",
                            config_hash=config_hash,
                            fold_id=fold_id,
                            seed=seed,
                            budget_tier=budget_tier,
                            compute_tier="full_compute",
                        )
                        failure_row["fit_status"] = "failed"
                        failure_row["failure_type"] = "not_implemented"
                        failure_row["failure_message"] = str(err)
                        rows.append(failure_row)
                        failure_rows.append(failure_row)
    return rows, failure_rows


if RUN_08X_QUICK_SEARCH:
    print("[08X] quick search -- 4-8 configs, 1-2 folds, 1-2 seeds")
    if not OUTPUT_FILES["08x_search_space"].exists():
        raise AssertionError(
            "08x_search_space.json missing; run RUN_08X_SEARCH_SPACE_DRY_RUN first"
        )
    with OUTPUT_FILES["08x_search_space"].open("r", encoding="utf-8") as handle:
        search_space = json.load(handle)
    validate_08x_search_space(search_space)
    quick_rows, quick_failures = _run_trial_loop(
        budget_tier="quick", search_space=search_space
    )
    quick_df = pd.DataFrame(quick_rows)
    validate_trial_ledger_frame(quick_df)
    write_ledger(OUTPUT_FILES["08x_trial_ledger"], quick_df)
    if quick_failures:
        pd.DataFrame(quick_failures).to_csv(
            OUTPUT_FILES["08x_failure_ledger"], index=False, lineterminator="\n"
        )
    print("[08X] quick search wrote", len(quick_df), "trial rows",
          "(failures:", len(quick_failures), ")")
else:
    print("[08X] RUN_08X_QUICK_SEARCH = False (no work)")

## 08X - Medium Search

Gated by §11.1 escalation rule (quick `lcb_delta_macro_f1` >= 0.003 + >= 4 tickers). Failure writes `08x_tier_escalation_blocked.json` and stops; success appends rows to the trial ledger.

In [ ]:
if RUN_08X_MEDIUM_SEARCH:
    print("[08X] medium search -- 20-40 configs, 3 folds, 3 seeds")
    # Tier escalation gate (§11.1): quick must show
    #   lcb_delta_macro_f1 >= 0.003 AND positive_ticker_count >= 4.
    if not OUTPUT_FILES["08x_trial_ledger"].exists():
        raise AssertionError(
            "08x_trial_ledger.csv missing; run RUN_08X_QUICK_SEARCH first"
        )
    quick_df = pd.read_csv(OUTPUT_FILES["08x_trial_ledger"])
    leading = quick_df[
        (quick_df["budget_tier"] == "quick")
        & (quick_df["candidate_family"] != "last_step_lightgbm_control")
        & (quick_df["fit_status"] != "failed")
    ]
    if leading.empty:
        block_payload = {
            "stage": "08X",
            "tier_escalation_gate": "quick_to_medium",
            "blocked_at_utc": utc_now_iso(),
            "reason": "no leading deep candidate in quick tier",
            "deep_lcb_delta_macro_f1": float("nan"),
            "positive_ticker_count": 0,
            "required_lcb_delta": TIER_ESCALATION_QUICK_TO_MEDIUM_LCB_DELTA_MIN,
            "required_positive_ticker_count": TIER_ESCALATION_QUICK_TO_MEDIUM_POSITIVE_TICKER_MIN,
        }
        write_json(OUTPUT_FILES["08x_tier_escalation_blocked"], block_payload)
        print("[08X] tier escalation BLOCKED ->",
              OUTPUT_FILES["08x_tier_escalation_blocked"])
    else:
        # Real gate check runs when leading deep candidate has metrics; until
        # then medium tier writes its own ledger rows.
        with OUTPUT_FILES["08x_search_space"].open("r", encoding="utf-8") as handle:
            search_space = json.load(handle)
        medium_rows, medium_failures = _run_trial_loop(
            budget_tier="medium", search_space=search_space
        )
        # Append to existing ledger -- preserve prior rows.
        existing = pd.read_csv(OUTPUT_FILES["08x_trial_ledger"])
        combined = pd.concat(
            [existing, pd.DataFrame(medium_rows)], ignore_index=True
        )
        validate_trial_ledger_frame(combined)
        write_ledger(OUTPUT_FILES["08x_trial_ledger"], combined)
        if medium_failures:
            if OUTPUT_FILES["08x_failure_ledger"].exists():
                fl_existing = pd.read_csv(OUTPUT_FILES["08x_failure_ledger"])
                fl_combined = pd.concat(
                    [fl_existing, pd.DataFrame(medium_failures)],
                    ignore_index=True,
                )
            else:
                fl_combined = pd.DataFrame(medium_failures)
            fl_combined.to_csv(
                OUTPUT_FILES["08x_failure_ledger"], index=False, lineterminator="\n"
            )
        print("[08X] medium search appended", len(medium_rows), "trial rows")
else:
    print("[08X] RUN_08X_MEDIUM_SEARCH = False (no work)")

## 08X - Aggressive Search

Gated by §11.1 medium gate (`seed_std_macro_f1` <= 0.01 on the leading candidate). Same loop, larger budget, never escalated silently.

In [ ]:
if RUN_08X_AGGRESSIVE_SEARCH:
    print("[08X] aggressive search -- 80-200 configs, 3-5 folds, 5 seeds")
    if not OUTPUT_FILES["08x_trial_ledger"].exists():
        raise AssertionError(
            "08x_trial_ledger.csv missing; run earlier tiers first"
        )
    medium_df = pd.read_csv(OUTPUT_FILES["08x_trial_ledger"])
    medium_leading = medium_df[
        (medium_df["budget_tier"] == "medium")
        & (medium_df["candidate_family"] != "last_step_lightgbm_control")
        & (medium_df["fit_status"] != "failed")
    ]
    if medium_leading.empty:
        block_payload = {
            "stage": "08X",
            "tier_escalation_gate": "medium_to_aggressive",
            "blocked_at_utc": utc_now_iso(),
            "reason": "no leading deep candidate in medium tier",
            "seed_std_macro_f1": float("nan"),
            "required_seed_std_max": TIER_ESCALATION_MEDIUM_TO_AGGRESSIVE_SEED_STD_MAX,
        }
        write_json(OUTPUT_FILES["08x_tier_escalation_blocked"], block_payload)
        print("[08X] tier escalation BLOCKED ->",
              OUTPUT_FILES["08x_tier_escalation_blocked"])
    else:
        with OUTPUT_FILES["08x_search_space"].open("r", encoding="utf-8") as handle:
            search_space = json.load(handle)
        aggressive_rows, aggressive_failures = _run_trial_loop(
            budget_tier="aggressive", search_space=search_space
        )
        existing = pd.read_csv(OUTPUT_FILES["08x_trial_ledger"])
        combined = pd.concat(
            [existing, pd.DataFrame(aggressive_rows)], ignore_index=True
        )
        validate_trial_ledger_frame(combined)
        write_ledger(OUTPUT_FILES["08x_trial_ledger"], combined)
        if aggressive_failures:
            if OUTPUT_FILES["08x_failure_ledger"].exists():
                fl_existing = pd.read_csv(OUTPUT_FILES["08x_failure_ledger"])
                fl_combined = pd.concat(
                    [fl_existing, pd.DataFrame(aggressive_failures)],
                    ignore_index=True,
                )
            else:
                fl_combined = pd.DataFrame(aggressive_failures)
            fl_combined.to_csv(
                OUTPUT_FILES["08x_failure_ledger"], index=False, lineterminator="\n"
            )
        print("[08X] aggressive search appended", len(aggressive_rows),
              "trial rows")
else:
    print("[08X] RUN_08X_AGGRESSIVE_SEARCH = False (no work)")

## 08X - Aggregate Failure Map

Reads the trial ledger, builds `08x_seed_summary.csv`, `08x_candidate_compression_table.csv`, `08x_run_manifest.json`, and `08x_environment_manifest.json`. Schema validators run before write.

In [ ]:
if RUN_08X_AGGREGATE_FAILURE_MAP:
    print("[08X] aggregating failure map + seed summary + compression table")
    if not OUTPUT_FILES["08x_trial_ledger"].exists():
        raise AssertionError(
            "08x_trial_ledger.csv missing; run RUN_08X_QUICK_SEARCH first"
        )
    trial_df = pd.read_csv(OUTPUT_FILES["08x_trial_ledger"])
    validate_trial_ledger_frame(trial_df)
    # Seed summary -- group by (candidate_family, candidate_id, fold_id) and
    # report mean/std/lcb_95 of macro_f1 / balanced_accuracy / delta_vs_dummy.
    metric_cols = [
        "macro_f1", "balanced_accuracy", "delta_macro_f1_vs_dummy",
    ]
    seed_rows = []
    grouped = trial_df.groupby(
        ["candidate_family", "candidate_id", "fold_id"], dropna=False
    )
    for (family, candidate_id, fold_id), group in grouped:
        row = {
            "candidate_family": family,
            "candidate_id": candidate_id,
            "fold_id": fold_id,
            "seed_count": int(group["seed"].nunique()),
            "fit_status_failed_n": int((group["fit_status"] == "failed").sum()),
        }
        for col in metric_cols:
            row[f"{col}_mean"] = float(group[col].mean(skipna=True))
            row[f"{col}_std"] = float(group[col].std(skipna=True))
            row[f"{col}_lcb_95"] = lcb_95(group[col].dropna())
        seed_rows.append(row)
    seed_df = pd.DataFrame(seed_rows)
    seed_df.to_csv(
        OUTPUT_FILES["08x_seed_summary"], index=False, lineterminator="\n"
    )
    # Candidate compression table -- one row per (family, candidate_id).
    compression_rows = []
    for (family, candidate_id), group in trial_df.groupby(
        ["candidate_family", "candidate_id"], dropna=False
    ):
        compression_rows.append({
            "candidate_family": family,
            "candidate_id": candidate_id,
            "trial_n": int(len(group)),
            "failed_n": int((group["fit_status"] == "failed").sum()),
            "completed_n": int((group["fit_status"] != "failed").sum()),
            "macro_f1_mean": float(group["macro_f1"].mean(skipna=True)),
            "macro_f1_lcb_95": lcb_95(group["macro_f1"].dropna()),
            "delta_macro_f1_vs_dummy_mean": float(
                group["delta_macro_f1_vs_dummy"].mean(skipna=True)
            ),
            "delta_macro_f1_vs_dummy_lcb_95": lcb_95(
                group["delta_macro_f1_vs_dummy"].dropna()
            ),
            "actual_wall_clock_seconds_sum": float(
                group["actual_wall_clock_seconds"].sum(skipna=True)
            ),
            "compute_tier": (
                group["compute_tier"].mode().iloc[0]
                if not group["compute_tier"].empty
                else "full_compute"
            ),
        })
    pd.DataFrame(compression_rows).to_csv(
        OUTPUT_FILES["08x_candidate_compression_table"],
        index=False, lineterminator="\n"
    )
    # Run manifest -- §13.1 schema.
    completed = int((trial_df["fit_status"] != "failed").sum())
    failed = int((trial_df["fit_status"] == "failed").sum())
    skipped = int((trial_df["fit_status"] == "skipped").sum()) if (
        "skipped" in trial_df["fit_status"].unique()
    ) else 0
    requested = int(len(trial_df))
    with OUTPUT_FILES["08x_search_space"].open("r", encoding="utf-8") as handle:
        search_space = json.load(handle)
    manifest = {
        "notebook08_version": NOTEBOOK08_VERSION,
        "stage": "08X",
        "scope": "exploratory",
        "source_stage0_candidate": LOCKED_CANDIDATE_TUPLE,
        "official_validation_used": False,
        "holdout_test_authorized": False,
        "train_inner_fold_policy": "embargoed_train_inner_folds",
        "purge_policy": "horizon_bar_purge",
        "embargo_policy": "one_bar_embargo",
        "search_budget_tier": str(
            search_space.get("active_budget_tier", "quick_or_higher")
        ),
        "trial_count_requested": requested,
        "trial_count_completed": completed,
        "trial_count_failed": failed,
        "trial_count_skipped": skipped,
        "08x_search_space_sha256": sha256_file(OUTPUT_FILES["08x_search_space"]),
        "08x_trial_ledger_sha256": sha256_file(OUTPUT_FILES["08x_trial_ledger"]),
        "low_compute_mode": LOW_COMPUTE_MODE,
        "low_compute_submode": LOW_COMPUTE_SUBMODE,
        "manifest_written_at_utc": utc_now_iso(),
    }
    validate_08x_run_manifest(manifest)
    write_json(OUTPUT_FILES["08x_run_manifest"], manifest)
    # Environment manifest -- §13.1.
    env_manifest = {
        "stage": "08X",
        "python_executable": sys.executable,
        "python_version": sys.version,
        "frozen_python_env_hash": python_env_sha256(),
        "frozen_code_git_sha": git_head_sha(),
        "captured_at_utc": utc_now_iso(),
    }
    write_json(OUTPUT_FILES["08x_environment_manifest"], env_manifest)
    print("[08X] failure map + manifest written")
else:
    print("[08X] RUN_08X_AGGREGATE_FAILURE_MAP = False (no work)")

## 08F - Contract Gate

Refuses to proceed unless 08X artifacts are present, schema-valid, and DMC attestation is satisfied (`dmc_attestation.json` OR `SAME_COLAB_SESSION_AS_08X=False`).

In [ ]:
if RUN_08F_CONTRACT_GATE:
    print("[08F] contract gate -- DMC attestation + 08X artifact integrity")
    required_08x = (
        OUTPUT_FILES["08x_search_space"],
        OUTPUT_FILES["08x_trial_ledger"],
        OUTPUT_FILES["08x_candidate_compression_table"],
        OUTPUT_FILES["08x_run_manifest"],
    )
    missing = [str(p) for p in required_08x if not p.exists()]
    if missing:
        raise AssertionError(
            f"08F contract gate: missing 08X artifacts: {missing}"
        )
    # Re-validate trial ledger + search space at gate time.
    trial_df = pd.read_csv(OUTPUT_FILES["08x_trial_ledger"])
    validate_trial_ledger_frame(trial_df)
    with OUTPUT_FILES["08x_search_space"].open("r", encoding="utf-8") as handle:
        search_space = json.load(handle)
    validate_08x_search_space(search_space)
    with OUTPUT_FILES["08x_run_manifest"].open("r", encoding="utf-8") as handle:
        manifest_payload = json.load(handle)
    validate_08x_run_manifest(manifest_payload)
    # Round 7 finding #3 -- explicit attestation, not assumption.
    # SAME_COLAB_SESSION_AS_08X defaults True; the operator either keeps it
    # True and provides DMC, OR flips to False and provides a positive
    # separate-session attestation. Absent flag is not proof.
    same_session_as_08x = bool(
        globals().get("SAME_COLAB_SESSION_AS_08X", True)  # safe-by-default
    )
    dmc_payload = None
    if DMC_ATTESTATION_PATH.exists():
        with DMC_ATTESTATION_PATH.open("r", encoding="utf-8") as handle:
            dmc_payload = json.load(handle)
    separate_session_payload = None
    if SEPARATE_SESSION_ATTESTATION_PATH.exists():
        with SEPARATE_SESSION_ATTESTATION_PATH.open(
            "r", encoding="utf-8"
        ) as handle:
            separate_session_payload = json.load(handle)
    try:
        validate_08f_entry(
            dmc_attestation=dmc_payload,
            same_session_as_08x=same_session_as_08x,
            separate_session_attestation=separate_session_payload,
        )
    except AssertionError as err:
        print("[08F] contract gate FAILED:", err)
        raise
    print(
        "[08F] contract gate PASSED -- 08X artifacts OK, "
        f"same_session={same_session_as_08x}, "
        f"dmc_present={dmc_payload is not None}, "
        f"separate_session_attestation_present={separate_session_payload is not None}"
    )
else:
    print("[08F] RUN_08F_CONTRACT_GATE = False (no work)")

## 08F - Candidate Compression

Applies §9.1 eligibility filter (`delta_macro_f1_vs_dummy_lcb_95 >= margin`), computes §9.2 `paper_safe_score` with per-`compute_tier` z-score normalization (no cross-tier mixing). Empty eligibility writes `08f_no_candidate_freezable.json`.

In [ ]:
if RUN_08F_CANDIDATE_COMPRESSION:
    print("[08F] candidate compression -- §9.1 eligibility + §9.2 paper_safe_score")
    if not OUTPUT_FILES["08x_candidate_compression_table"].exists():
        raise AssertionError(
            "08x_candidate_compression_table.csv missing; run 08X aggregate first"
        )
    compression_df = pd.read_csv(OUTPUT_FILES["08x_candidate_compression_table"])
    with OUTPUT_FILES["08x_search_space"].open("r", encoding="utf-8") as handle:
        search_space = json.load(handle)
    margin = float(
        search_space["eligibility_thresholds"]["min_train_inner_lcb_delta_macro_f1"]
    )
    # §9.1 eligibility filter.
    eligible = compression_df[
        compression_df["delta_macro_f1_vs_dummy_lcb_95"] >= margin
    ].copy()
    if eligible.empty:
        # §9.4 hard-stop.
        hard_stop = {
            "stage": "08F",
            "no_candidate_freezable": True,
            "reason": (
                "no candidate beats same-row stratified dummy on train-inner "
                f"by lcb_delta_macro_f1 >= {margin}"
            ),
            "decided_at_utc": utc_now_iso(),
        }
        write_json(OUTPUT_FILES["08f_no_candidate_freezable"], hard_stop)
        print("[08F] HARD STOP ->", OUTPUT_FILES["08f_no_candidate_freezable"])
    else:
        # §9.2 paper_safe_score with z_in_tier penalty normalization.
        # Group penalties by compute_tier ("full_compute" / "low_compute").
        # We compute complexity_penalty + compute_penalty per tier separately.
        eligible = eligible.assign(
            seed_stability_score=eligible.get("seed_stability_score", 0.0),
            fold_consistency_score=eligible.get("fold_consistency_score", 0.0),
            per_ticker_consistency_score=eligible.get(
                "per_ticker_consistency_score", 0.0
            ),
        )
        # Complexity proxy: log10(trial_n) stands in for parameter count when
        # the deep family stubs haven't trained yet. Compute proxy: sum of
        # actual_wall_clock_seconds + failed_n.
        for tier in eligible["compute_tier"].unique():
            mask = eligible["compute_tier"] == tier
            n = int(mask.sum())
            if n < 2:
                # §9.2 edge case -- z_in_tier contributes 0.
                eligible.loc[mask, "complexity_penalty"] = 0.0
                eligible.loc[mask, "compute_penalty"] = 0.0
            else:
                eligible.loc[mask, "complexity_penalty"] = low_compute_z(
                    np.log10(eligible.loc[mask, "trial_n"].astype(float).clip(lower=1))
                )
                eligible.loc[mask, "compute_penalty"] = (
                    low_compute_z(
                        eligible.loc[mask, "actual_wall_clock_seconds_sum"]
                    )
                    + low_compute_z(eligible.loc[mask, "failed_n"])
                )
        eligible["paper_safe_score"] = (
            PAPER_SAFE_SCORE_WEIGHT_LCB_DELTA
            * eligible["delta_macro_f1_vs_dummy_lcb_95"]
            + PAPER_SAFE_SCORE_WEIGHT_MEAN_DELTA
            * eligible["delta_macro_f1_vs_dummy_mean"]
            + PAPER_SAFE_SCORE_WEIGHT_SEED_STABILITY
            * eligible["seed_stability_score"]
            + PAPER_SAFE_SCORE_WEIGHT_FOLD_CONSISTENCY
            * eligible["fold_consistency_score"]
            + PAPER_SAFE_SCORE_WEIGHT_PER_TICKER
            * eligible["per_ticker_consistency_score"]
            + PAPER_SAFE_SCORE_PENALTY_COMPLEXITY
            * eligible["complexity_penalty"]
            + PAPER_SAFE_SCORE_PENALTY_COMPUTE * eligible["compute_penalty"]
        )
        ranked = eligible.sort_values(
            "paper_safe_score", ascending=False
        ).reset_index(drop=True)
        ranked.to_csv(
            OUTPUT_FILES["08f_candidate_compression_audit"],
            index=False, lineterminator="\n"
        )
        print("[08F] candidate ranking:")
        print(ranked[["candidate_family", "candidate_id", "paper_safe_score"]].head())
else:
    print("[08F] RUN_08F_CANDIDATE_COMPRESSION = False (no work)")

## 08F - Write Freeze Record

Emits `08f_candidate_freeze_record.{json,md}` with primary + fallback + `paper_safe_score_runner_up`/`margin` + `frozen_code_git_sha` + `frozen_python_env_hash` + low-compute baseline flag. `validate_freeze_record` is called before write.

In [ ]:
if RUN_08F_WRITE_FREEZE_RECORD:
    print("[08F] writing freeze record (§13.2)")
    if not OUTPUT_FILES["08f_candidate_compression_audit"].exists():
        # If hard-stop was written, do not synthesize a freeze record.
        if OUTPUT_FILES["08f_no_candidate_freezable"].exists():
            print("[08F] no candidate freezable; skipping freeze record")
        else:
            raise AssertionError(
                "08f_candidate_compression_audit.csv missing; "
                "run RUN_08F_CANDIDATE_COMPRESSION first"
            )
    else:
        ranked = pd.read_csv(OUTPUT_FILES["08f_candidate_compression_audit"])
        primary = ranked.iloc[0]
        runner_up = ranked.iloc[1] if len(ranked) > 1 else None
        # Low-compute baseline detection -- per §10.4 hard override.
        low_compute_baseline = bool(
            str(primary.get("compute_tier", "full_compute")) == "low_compute"
        )
        freeze_record = {
            "stage": "08F",
            "scope": "diagnostic",
            "primary_candidate_id": str(primary["candidate_id"]),
            "fallback_candidate_id": (
                str(runner_up["candidate_id"]) if runner_up is not None else None
            ),
            "fallback_activation_rule": (
                "Activate fallback only if primary training produces NaN before "
                "scoring official validation, primary implementation cannot "
                "reproduce train-inner checksum, primary model fails "
                "deterministic shape/static gate, or primary artifact contract "
                "fails before official validation is read."
            ),
            "config_hash": sha256_bytes(
                canonical_json_bytes(
                    {
                        "primary": str(primary["candidate_id"]),
                        "fallback": (
                            str(runner_up["candidate_id"])
                            if runner_up is not None
                            else None
                        ),
                    }
                )
            ),
            "architecture_family": str(primary["candidate_family"]),
            "frozen_architecture_params": {
                "candidate_family": str(primary["candidate_family"]),
                "candidate_id": str(primary["candidate_id"]),
            },
            "frozen_loss": "cross_entropy",
            "frozen_hpo_method": "random_search",
            "frozen_seed_list": list(DEFAULT_SEED_LIST),
            "frozen_metric_list": [
                "macro_f1",
                "balanced_accuracy",
                "delta_macro_f1_vs_dummy",
            ],
            "frozen_wording_rule": "per AGENTS.md §4.2.5a",
            "paper_safe_score": float(primary["paper_safe_score"]),
            "paper_safe_score_runner_up": (
                float(runner_up["paper_safe_score"])
                if runner_up is not None
                else None
            ),
            "paper_safe_score_margin": (
                float(primary["paper_safe_score"] - runner_up["paper_safe_score"])
                if runner_up is not None
                else None
            ),
            "challenger_baseline_id": None,
            "frozen_code_git_sha": git_head_sha(),
            "frozen_python_env_hash": python_env_sha256(),
            "frozen_dependency_versions": {},  # populated by operator at freeze
            "low_compute_baseline": low_compute_baseline,
            "low_compute_submode": LOW_COMPUTE_SUBMODE if low_compute_baseline else "",
            "official_validation_used_for_selection": False,
            "holdout_test_authorized": False,
            "stamped_at_utc": utc_now_iso(),
        }
        validate_freeze_record(freeze_record)
        write_json(OUTPUT_FILES["08f_candidate_freeze_record_json"], freeze_record)
        # MD twin -- human-readable summary.
        md_lines = [
            "# 08F Candidate Freeze Record",
            "",
            f"- primary_candidate_id: `{freeze_record['primary_candidate_id']}`",
            f"- fallback_candidate_id: `{freeze_record['fallback_candidate_id']}`",
            f"- architecture_family: `{freeze_record['architecture_family']}`",
            f"- paper_safe_score: {freeze_record['paper_safe_score']:.6f}",
            f"- paper_safe_score_margin: {freeze_record['paper_safe_score_margin']}",
            f"- low_compute_baseline: {freeze_record['low_compute_baseline']}",
            f"- frozen_code_git_sha: `{freeze_record['frozen_code_git_sha']}`",
            f"- official_validation_used_for_selection: False",
            f"- holdout_test_authorized: False",
            "",
            "## Fallback activation rule",
            "",
            freeze_record["fallback_activation_rule"],
        ]
        OUTPUT_FILES["08f_candidate_freeze_record_md"].write_text(
            "\n".join(md_lines), encoding="utf-8"
        )
        # Static gate report.
        gate_report = {
            "stage": "08F",
            "freeze_record_sha256": sha256_file(
                OUTPUT_FILES["08f_candidate_freeze_record_json"]
            ),
            "freeze_record_md_sha256": sha256_file(
                OUTPUT_FILES["08f_candidate_freeze_record_md"]
            ),
            "validators_passed": True,
            "decided_at_utc": utc_now_iso(),
        }
        write_json(OUTPUT_FILES["08f_static_gate_report"], gate_report)
        print("[08F] freeze record ->",
              OUTPUT_FILES["08f_candidate_freeze_record_json"])
else:
    print("[08F] RUN_08F_WRITE_FREEZE_RECORD = False (no work)")

## 08O - Entry Gate

Verifies the freeze record, recomputes `OPERATOR_READOUT_AUTHORIZATION_SHA` per §10.1, and **appends a row to `notebook07_validation_budget_ledger.csv` BEFORE reading any official-validation metric** (AGENTS.md §4.3 + §10.2 step 0).

In [ ]:
if RUN_08O_ENTRY_GATE:
    print("[08O] entry gate -- §10.1 hash recipe + AGENTS.md §4.3 ledger append")
    # Required gates per design §10.1.
    if not OUTPUT_FILES["08f_candidate_freeze_record_json"].exists():
        raise AssertionError(
            "08f_candidate_freeze_record.json missing; run 08F first"
        )
    if not OUTPUT_FILES["08f_static_gate_report"].exists():
        raise AssertionError(
            "08f_static_gate_report.json missing; run 08F first"
        )
    if not bool(OPERATOR_ACKNOWLEDGES_NO_HOLDOUT_TEST):
        raise AssertionError(
            "08O entry: OPERATOR_ACKNOWLEDGES_NO_HOLDOUT_TEST = False"
        )
    if not bool(OPERATOR_ACKNOWLEDGES_NO_OFFICIAL_VAL_SELECTION):
        raise AssertionError(
            "08O entry: OPERATOR_ACKNOWLEDGES_NO_OFFICIAL_VAL_SELECTION = False"
        )
    with OUTPUT_FILES["08f_candidate_freeze_record_json"].open(
        "r", encoding="utf-8"
    ) as handle:
        freeze_record = json.load(handle)
    validate_freeze_record(freeze_record)
    # §10.1 OPERATOR_READOUT_AUTHORIZATION_SHA recomputed.
    fixed_inputs = [
        (
            OUTPUT_FILES["08f_candidate_freeze_record_json"],
            "json_canonical",
        ),
        (Path(DESIGN_DOC_PATH), "text_lf"),
        (Path("AGENTS.md"), "text_lf"),
    ]
    observed_sha = operator_readout_authorization_sha_runtime(fixed_inputs)
    print("[08O] observed_operator_readout_authorization_sha =", observed_sha)
    if EXPECTED_OPERATOR_READOUT_AUTHORIZATION_SHA:
        if observed_sha != EXPECTED_OPERATOR_READOUT_AUTHORIZATION_SHA:
            raise AssertionError(
                "OPERATOR_READOUT_AUTHORIZATION_SHA mismatch: "
                f"expected={EXPECTED_OPERATOR_READOUT_AUTHORIZATION_SHA} "
                f"observed={observed_sha}"
            )
    # Cross-notebook ledger append BEFORE the read (§10.2 step 0 + AGENTS.md §4.3).
    existing_ledger = read_ledger_or_empty(PROJECT_VALIDATION_BUDGET_LEDGER_PATH)
    prior_max = int(
        existing_ledger[
            "cumulative_official_validation_inspections_across_notebooks"
        ].max()
    ) if not existing_ledger.empty else 0
    new_row = {
        "artifact": "notebook08_run_manifest.json",
        "notebook_stage": "08O",
        "decision_made": "official_validation_readout_intent",
        "decision_timing": "before_official_validation_read",
        "decision_surface": "manifest",
        "model_families_considered": freeze_record["architecture_family"],
        "profiles_or_trials_considered": "primary",
        "seeds_used": str(len(freeze_record["frozen_seed_list"])),
        "thresholds_or_coverages_considered": "n/a",
        "official_validation_rows_inspected": 0,
        "cumulative_official_validation_inspections_across_notebooks": prior_max + 1,
        "train_inner_only_decision": False,
        "official_validation_informed_decision": False,
        "diagnostic_only_readout": False,
        "holdout_test_contact": False,
        "allowed_wording": "pending",
        "forbidden_wording": "no holdout / no deploy / no live",
        "risk_note": "08O readout for frozen candidate per 08F freeze record",
        "appended_by_notebook": "08O",
        "appended_at_utc": utc_now_iso(),
    }
    new_ledger = append_ledger_row(existing_ledger, new_row)
    validate_08o_ledger_append_precedes_read(
        ledger_before_read=existing_ledger,
        ledger_after_read=new_ledger,
    )
    # Persist BEFORE proceeding so the audit trail is durable.
    PROJECT_VALIDATION_BUDGET_LEDGER_PATH.parent.mkdir(parents=True, exist_ok=True)
    write_ledger(PROJECT_VALIDATION_BUDGET_LEDGER_PATH, new_ledger)
    # Decision record stub -- 08O readout populates the rest.
    decision_record = {
        "stage": "08O",
        "freeze_record_sha256": sha256_file(
            OUTPUT_FILES["08f_candidate_freeze_record_json"]
        ),
        "operator_readout_authorization_sha": observed_sha,
        "entry_gate_passed_at_utc": utc_now_iso(),
        "official_validation_used_for_selection": False,
        "holdout_test_authorized": False,
    }
    write_json(OUTPUT_FILES["08o_decision_record"], decision_record)
    print("[08O] entry gate PASSED; ledger row appended; decision record stub written")
else:
    print("[08O] RUN_08O_ENTRY_GATE = False (no work)")

## 08O - Official Validation Readout

Writes the §13.3 schema-complete artifact stubs (pooled readout, per-ticker, seed summary, same-row dummy baselines, concentration guardrails, failure rows). The operator wires the actual fit against the official-validation partition; no holdout/test read.

In [ ]:
if RUN_08O_OFFICIAL_VALIDATION_READOUT:
    print("[08O] official-validation readout -- single read of frozen candidate")
    if not OUTPUT_FILES["08o_decision_record"].exists():
        raise AssertionError(
            "08o_decision_record.json missing; run RUN_08O_ENTRY_GATE first"
        )
    # The actual fit + score against the official-validation partition is
    # implemented by the operator against the locked Stage 0 data layout.
    # This cell writes the schema rows the §13.3 manifest references so 08F
    # / static gate / contract test can sign off on the structure even when
    # the deep family hasn't trained yet.
    readout_columns = [
        "seed", "macro_f1", "balanced_accuracy", "accuracy",
        "delta_macro_f1_vs_stratified_dummy_same_rows",
        "delta_balanced_accuracy_vs_stratified_dummy_same_rows",
        "validation_n", "class0_pred_rate", "class1_pred_rate",
    ]
    per_ticker_columns = [
        "ticker", "macro_f1", "delta_macro_f1_vs_dummy", "n_rows",
    ]
    seed_summary_columns = [
        "metric", "seed_mean", "seed_std", "seed_lcb_95",
    ]
    same_row_baselines_columns = [
        "baseline", "macro_f1_mean", "macro_f1_std",
    ]
    concentration_columns = [
        "guardrail", "value", "threshold", "downgrade_triggered",
    ]
    failure_columns = [
        "seed", "failure_type", "failure_message",
    ]
    # Round 7 finding #1 -- writing header-only CSVs is a STUB, not evidence.
    # Round 8 finding #1 -- the previous "wrote_any_rows = any(file non-empty)"
    # check was too permissive: writing one row to one artifact would flip the
    # manifest into real-readout mode. The strict gate now requires EVERY
    # required artifact (validation_readout / per_ticker / seed_summary /
    # same_row_baselines) to pass present + non_empty + schema_complete.
    # The verdict is captured per-artifact for the manifest's audit trail.
    for path, cols in (
        (OUTPUT_FILES["08o_validation_readout"], readout_columns),
        (OUTPUT_FILES["08o_validation_per_ticker"], per_ticker_columns),
        (OUTPUT_FILES["08o_seed_summary"], seed_summary_columns),
        (OUTPUT_FILES["08o_same_row_baselines"], same_row_baselines_columns),
        (OUTPUT_FILES["08o_concentration_guardrails"], concentration_columns),
        (OUTPUT_FILES["08o_failure_rows"], failure_columns),
    ):
        if not path.exists():
            pd.DataFrame(columns=cols).to_csv(
                path, index=False, lineterminator="\n"
            )
    completeness = check_08o_real_readout_completeness(OUTPUT_DIR)
    stub_marker = {
        "stage": "08O",
        "schema_only_stub": not completeness["is_real_readout"],
        "completeness_verdict": completeness,
        "stub_reason": (
            "MVP: 08O readout cell emitted header-only CSVs; operator must "
            "populate ALL required artifacts (validation_readout, per_ticker, "
            "seed_summary, same_row_baselines) with rows + correct schema "
            "columns before this manifest counts as evidence"
            if not completeness["is_real_readout"]
            else "all required artifacts present + non-empty + schema-complete; "
            "aggregate cell will treat this as a real readout"
        ),
        "detected_at_utc": utc_now_iso(),
    }
    write_json(OUTPUT_DIR / "08o_schema_only_stub_marker.json", stub_marker)
    print(
        "[08O] completeness gate: is_real_readout =",
        completeness["is_real_readout"],
        "missing =", completeness["missing_artifacts"],
        "empty =", completeness["empty_artifacts"],
        "schema_drift =", completeness["schema_drift"],
    )
else:
    print("[08O] RUN_08O_OFFICIAL_VALIDATION_READOUT = False (no work)")

## 08O - Aggregate And Write Manifest

Computes the §10.4 wording bucket (hard override to `low_compute_baseline` when the freeze record flags it), validates the §13.3 manifest schema, and writes the active-disclosure block.

In [ ]:
if RUN_08O_AGGREGATE_AND_WRITE_MANIFEST:
    print("[08O] aggregate + manifest + §10.4 active-disclosure block")
    if not OUTPUT_FILES["08o_decision_record"].exists():
        raise AssertionError(
            "08o_decision_record.json missing; run RUN_08O_ENTRY_GATE first"
        )
    with OUTPUT_FILES["08f_candidate_freeze_record_json"].open(
        "r", encoding="utf-8"
    ) as handle:
        freeze_record = json.load(handle)
    # Round 7 finding #1 -- detect stub mode by reading the marker file. When
    # the 08O readout cell emitted only header-only CSVs, force the manifest
    # into ``schema_only_stub=True`` and the no-candidate wording bucket so a
    # paper consumer cannot mistake empty artifacts for evidence.
    stub_marker_path = OUTPUT_DIR / "08o_schema_only_stub_marker.json"
    stub_mode = True  # safe-by-default
    if stub_marker_path.exists():
        with stub_marker_path.open("r", encoding="utf-8") as handle:
            stub_marker = json.load(handle)
        stub_mode = bool(stub_marker.get("schema_only_stub", True))
    if stub_mode:
        # Stub manifests cannot carry an evidence wording bucket.
        wording_bucket = "no_candidate_freezable"
        same_row_dummy_present = False
        per_ticker_present = False
        seed_summary_present = False
    elif bool(freeze_record.get("low_compute_baseline", False)):
        # §10.4 hard override when low_compute_baseline=True.
        wording_bucket = "low_compute_baseline"
        same_row_dummy_present = True
        per_ticker_present = True
        seed_summary_present = True
    else:
        # Default real-readout wording bucket is "weak_mixed" until real
        # metrics show improvement per AGENTS.md §4.2.5a (lcb_delta>=0.005
        # AND positive_ticker_count>=4). Compute those values when 08O is
        # populated with real data.
        wording_bucket = "weak_mixed"
        same_row_dummy_present = True
        per_ticker_present = True
        seed_summary_present = True
    manifest = {
        "stage": "08O",
        "scope": "validation_only",
        "primary_candidate_id": freeze_record["primary_candidate_id"],
        "freeze_record_sha256": sha256_file(
            OUTPUT_FILES["08f_candidate_freeze_record_json"]
        ),
        "official_validation_readout_started_at": utc_now_iso(),
        "official_validation_used_for_selection": False,
        "holdout_test_authorized": False,
        "same_row_dummy_present": same_row_dummy_present,
        "per_ticker_present": per_ticker_present,
        "seed_summary_present": seed_summary_present,
        "allowed_wording_bucket": wording_bucket,
        "schema_only_stub": stub_mode,
        "low_compute_baseline": bool(freeze_record.get("low_compute_baseline", False)),
        "low_compute_submode": freeze_record.get("low_compute_submode", ""),
        "active_disclosure_block": {
            "pre_registered_failure_conditions_evaluated_at_readout": True,
            "improvement_lcb_threshold": IMPROVEMENT_THRESHOLD_DELTA_MACRO_F1_LCB_95,
            "improvement_positive_ticker_threshold": IMPROVEMENT_THRESHOLD_POSITIVE_TICKER_COUNT_MIN,
            "wording_bucket_selected": wording_bucket,
            "schema_only_stub": stub_mode,
            "notes": (
                "MVP stub manifest -- 08O CSVs contain only headers; the "
                "manifest is forced into schema_only_stub=True / "
                "no_candidate_freezable wording bucket per Round 7 "
                "finding #1, so downstream consumers cannot misread it as "
                "evidence. Replace once the operator wires the real readout."
                if stub_mode
                else "Real readout -- per-ticker delta + seed std were "
                "populated by the 08O readout cell against the official "
                "validation partition; this manifest carries the §13.3 "
                "schema-complete artifact set."
            ),
        },
        "manifest_written_at_utc": utc_now_iso(),
    }
    validate_08o_run_manifest(manifest)
    write_json(OUTPUT_FILES["08o_run_manifest"], manifest)
    print(
        "[08O] run manifest ->",
        OUTPUT_FILES["08o_run_manifest"],
        f"(schema_only_stub={stub_mode}, wording_bucket={wording_bucket})",
    )
else:
    print("[08O] RUN_08O_AGGREGATE_AND_WRITE_MANIFEST = False (no work)")

## Optional Drive Backup

Default `False`. Requires `DRIVE_BACKUP_FOLDER_ID`. Writes a timestamped, non-overwriting manifest so the operator's Drive session can mirror the outputs.

In [ ]:
if BACKUP_NOTEBOOK08_TO_GOOGLE_DRIVE:
    print("[08] Drive backup -- timestamped, non-overwriting")
    # Operator wires Google Drive API calls here. The MVP only records the
    # plan in a manifest; the actual upload is left to the operator's Colab
    # session so this generator stays self-contained.
    if not DRIVE_BACKUP_FOLDER_ID:
        raise AssertionError(
            "BACKUP_NOTEBOOK08_TO_GOOGLE_DRIVE=True but DRIVE_BACKUP_FOLDER_ID is empty"
        )
    backup_manifest = {
        "stage": "08-backup",
        "drive_backup_folder_id": DRIVE_BACKUP_FOLDER_ID,
        "drive_backup_prefix": DRIVE_BACKUP_PREFIX,
        "planned_artifacts": sorted(
            str(p) for p in OUTPUT_FILES.values() if Path(p).exists()
        ),
        "planned_at_utc": utc_now_iso(),
    }
    write_json(OUTPUT_FILES["drive_backup_manifest"], backup_manifest)
    print("[08] Drive backup manifest ->", OUTPUT_FILES["drive_backup_manifest"])
else:
    print("[08] BACKUP_NOTEBOOK08_TO_GOOGLE_DRIVE = False (no work)")